Connected to olympics (Python 3.11.15)

In [ ]:
"""
Author: Sydney Smith
Date Created: September 29, 2026
"""

from datetime import datetime
import gc
import glob
from loguru import logger
import numpy as np
import os
import pandas as pd
from pathlib import Path
from scipy.interpolate import interp1d
from statsmodels.distributions.empirical_distribution import ECDF, monotone_fn_inverter
import sys
import time
import xarray as xr

# ==================================
# - Establish Relative File Path - 
# ==================================

current_dir = Path(__file__).resolve().parent
sys.path.append(str(current_dir))

from plots.validation import running_window_slice, loc_sel

# ===================
# - Set Up Logger - 
# ==================

# Create directory for log files if it doesn't already exist
log_path = str(current_dir / 'log')
os.makedirs(log_path, exist_ok = True)

# String filename
log_filename = datetime.now().strftime('log/%Y-%m-%d_%H-%M-%S.log')

# Custom format for log prints
log_format = log_format = '<cyan>{time:HH:mm:ss}</cyan> | <level>{level:>8}</level> | <yellow>{name}</yellow>:<cyan>{function}</cyan>:line <magenta>{line}</magenta> - <level>{message}</level>'

# Removes the default stderr sink
logger.remove()  
# Anything above info to console
logger.add(sys.stderr, colorize = True, format = log_format, level = 'INFO') 
# Anything above debug to log file
logger.add(log_filename, colorize = False, format = log_format, level = 'DEBUG')
logger.debug(f'Log files saved to {log_path}')

# =====================
# ---- Functions ----
# =====================

def invert_ecdf(ecdf_obj):
    """
    Inverts a statsmodels ECDF object using its .y and .x attributes, regardless of if
    the values are monotonic or not, with safe extrapolation to prevent NaNs.
    """
    return interp1d(
        ecdf_obj.y, # Pass y values as x values
        ecdf_obj.x, # And vice versa
        bounds_error = False, 
        fill_value = 'extrapolate'  # Prevents NaNs if raw percentiles push past bounds
    )

def apply_to_date(obs: xr.Dataset, hist, raw, var, date) -> np.ndarray:
    """
    Debiases data for a given date by constructing a CDF using a dataset restricted to be within a 
    running window. Date must be a valid date that exists within the dataset formatted as '%Y-%m-%d' 
    datetime object.
    """
    # TODO: check how running_window_slice handles leap years

    # Slice data into 31 day running windows centered on given date (note: 31 days is defualt setting)
    obs_sliced = running_window_slice(obs, date)
    logger.info(f'NaNs in obs running window: {float(obs_sliced[var].isnull().sum().values)}')
    hist_sliced = running_window_slice(hist, date)
    logger.info(f'NaNs in hist running window: {float(hist_sliced[var].isnull().sum().values)}')
    raw_sliced = running_window_slice(raw, date)
    logger.info(f'NaNs in raw running window: {float(raw_sliced[var].isnull().sum().values)}')
    logger.info('Datasets sliced into running windows.')

    # Create CDFs from sliced datasets
    obs_cdf = ECDF(obs_sliced[var].values.flatten())
    hist_cdf = ECDF(hist_sliced[var].values.flatten())
    raw_cdf = ECDF(raw_sliced[var].values.flatten())

    # Select day of interest and create mask
    day = int(date.strftime('%d'))
    month = int(date.strftime('%m'))
    mask = (raw.time.dt.month == month) & (raw.time.dt.day == day)

    # Skip dates that don't exist
    if not mask.any():
        logger.warning(f'No date found for {date}.')
        return None
    
    # Apply mask to dataset
    raw_date = raw.where(mask, drop = True) # This should select day of interest across all years in ds
    logger.info(f'NaNs in raw data for {date}: {float(raw_date[var].isnull().sum().values)}')
    logger.info(f'Dataset sliced down to date of interest: {date}')
    logger.info(raw_date)

    # Use raw_cdf like a function to find all of the percentiles in raw_date
    raw_percentiles = raw_cdf(raw_date[var].values)
    logger.info(f'Percentile values for raw data: {raw_percentiles}')
    logger.info(f'NaNs in raw_percentiles: {float(np.isnan(raw_percentiles).sum())}')

    # TODO: NaNs are appearing in obs and hist from this point on
    # Invert hist_cdf and obs_cdf to take a percentile and output a data point
    inv_obs = invert_ecdf(obs_cdf)
    inv_hist = invert_ecdf(hist_cdf)

    # Find data points for every percentile in raw_percentiles
    obs_vals = inv_obs(raw_percentiles)
    logger.info(f'NaNs in obs_vals: {float(np.isnan(obs_vals).sum())}')
    logger.info(f'Obs values for given percentiles: {obs_vals}')
    hist_vals = inv_hist(raw_percentiles)
    logger.info(f'NaNs in hist_vals: {float(np.isnan(hist_vals).sum())}')
    logger.info(f'Historical values for given percentiles: {hist_vals}')

    # Perform bias adjustment
    bias = hist_vals - obs_vals
    logger.info(f'NaNs in bias: {float(np.isnan(bias).sum())}')
    raw_bias_corrected = raw_date[var].values - bias

    # End script early if there are Nans in data
    if np.isnan(raw_bias_corrected).any():
        logger.error(f'NaNs in debiased data: {float(np.isnan(raw_bias_corrected).sum())}')
        logger.info(f'Nan error occured on {date}.')
        return None

    logger.info('Raw data successfully bias corrected!')
    logger.info(raw_bias_corrected)
    logger.info(f'Bias corrected max: {raw_bias_corrected.max()}')
    logger.info(f'Bias corrected min: {raw_bias_corrected.min()}')

    # Close out of unwanted datasets
    obs_sliced.close()
    hist_sliced.close()
    raw_sliced.close()
    raw_date.close()
    del obs_vals
    del hist_vals
    del raw_percentiles
    gc.collect()

    return raw_bias_corrected

def ECDFM(obs, raw, var, lat, lon):
    """
    Performs ECDFM quantile mapping debiasing process on given data based on data from a given histroical
    period and corresponding observational data. Raw data should include both historical and future data. 
    """

    # Slice raw data down to historical period
    hist = raw.sel(time = slice('1985-01-01', '2014-12-31'))
    
    # Select specific location of interest (choses nearest possible location)
    obs_loc = loc_sel(obs, lat = lat, lon = lon)
    logger.info(f'NaNs at selected obs location: {float(obs_loc[var].isnull().sum().values)}')
    hist_loc = loc_sel(hist, lat = lat, lon = lon)
    logger.info(f'NaNs at selected hist location: {float(hist_loc[var].isnull().sum().values)}')
    raw_loc = loc_sel(raw, lat = lat, lon = lon)
    logger.info(f'NaNs at selected raw location: {float(raw_loc[var].isnull().sum().values)}')
    logger.info(raw_loc[var])

    # Make copy of raw_loc to plug debiased data into
    template = raw_loc.copy()

    # Create date range that spans a calendar year
    dates = pd.date_range(start = '1985-01-01', end = '1985-12-31', freq = 'D') 

    for date in dates:
        # Apply bias correction for single date
        logger.info(f'Date type: {type(date)}')
        debiased_doy = apply_to_date(obs_loc, hist_loc, raw_loc, var, date)

        # Exit out of function if Nans occur
        if debiased_doy is None:
            logger.error(f'Nans detected in dataset. See log above for more information.')
            return None

        # Pull month and day from date
        day = int(date.strftime('%d'))
        month = int(date.strftime('%m'))

        # Plug in debiased data
        mask = (template.time.dt.month == month) & (template.time.dt.day == day)
        template[var].data[mask] = debiased_doy 
        logger.info(f'Debiased data for {date} added to dataset!')

        # Check that data was actaully replaced
        temp_masked = template[var].where(mask, drop = True)
        raw_masked = raw[var].where(mask, drop = True)
        bias = raw_masked - temp_masked # Back into data's bias
        if (bias.all() == 0): # Ensure it's not all zeros
            logger.error('Bias corrected data was not successfully applied to dataset.')
            return None
        else:
            logger.info(f'Bias for {date} successfully corrected.')

        # Close unnecessay datasets
        temp_masked.close()
        raw_masked.close()

    return template

def nan_test(var):
    """
    Returns location of Nans in debiased data.
    """
    
    fpath = glob.glob(str(current_dir / 'wrfout' / f'*{var}*.nc'))
    debiased = xr.open_dataset(fpath[0], decode_times = True)

    nan_data = debiased[var].where(debiased[var].isnull(), drop = True)

    return nan_data

# Catch silent errors and report to log file
@logger.catch 
def main(var, data_location, save = True):
    # Open datasets
    obs_path = glob.glob(str(data_location / 'gridMET' / f'*{var}*.nc'))
    obs = xr.open_dataset(obs_path[0], decode_times = True)

    raw_path = glob.glob(str(data_location / 'daily' / f'*{var}*.nc'))
    raw = xr.open_dataset(raw_path[0], decode_times = True)

    # Set location of interest as SLC airport
    lat = 40.788
    lon = -111.978

    # Pass datasets to debiaser
    test = ECDFM(obs, raw, var, lat, lon)

    if save:
        # Create output directory to store new cleaned files
        output_dir = current_dir / 'wrfout' 
        os.makedirs(output_dir, exist_ok = True) # Don't make if it already exists

        # Generate save name for debiased data
        out_path = os.path.join(output_dir, f'wrfout_GSLBIP_multimodel_ssp245_{var}_test.nc')

        # Save data to netCDF file
        test.to_netcdf(out_path)
        logger.success(f'File saved to: {out_path}')

    # Close out of data
    test.close()

# if __name__ == '__main__':

#     # Track program time in log files
#     start = time.perf_counter()
#     logger.info('Beginning execution.')

#     # Only inputs required
#     main(
#             var = 'tmmx',
#             data_location = current_dir,
#             save = False
#             )

#     # Report of runtime at completion 
#     logger.success(f'Debiasing process completed!')
#     logger.info(f'Total runtime: {time.perf_counter() - start:.4f}s')

#     # Force script to stop running once code is finished
#     sys.exit(0)

var = 'tmmx'
# Open datasets
obs_path = glob.glob(str(current_dir / 'gridMET' / f'*{var}*.nc'))
obs = xr.open_dataset(obs_path[0], decode_times = True)

raw_path = glob.glob(str(current_dir / 'daily' / f'*{var}*.nc'))
raw = xr.open_dataset(raw_path[0], decode_times = True)

# Set location of interest as SLC airport
lat = 40.788
lon = -111.978

# Slice raw data down to historical period
hist = raw.sel(time = slice('1985-01-01', '2014-12-31'))

# Select specific location of interest (choses nearest possible location)
obs_loc = loc_sel(obs, lat = lat, lon = lon)
logger.info(f'NaNs at selected obs location: {float(obs_loc[var].isnull().sum().values)}')
hist_loc = loc_sel(hist, lat = lat, lon = lon)
logger.info(f'NaNs at selected hist location: {float(hist_loc[var].isnull().sum().values)}')
raw_loc = loc_sel(raw, lat = lat, lon = lon)
logger.info(f'NaNs at selected raw location: {float(raw_loc[var].isnull().sum().values)}')
logger.info(raw_loc[var])

# Make copy of raw_loc to plug debiased data into
template = raw_loc.copy()

# Create date range that spans a calendar year
date = pd.to_datetime('1985-01-16') 

# Slice data into 31 day running windows centered on given date (note: 31 days is defualt setting)
obs_sliced = running_window_slice(obs, date)
logger.info(f'NaNs in obs running window: {float(obs_sliced[var].isnull().sum().values)}')
hist_sliced = running_window_slice(hist, date)
logger.info(f'NaNs in hist running window: {float(hist_sliced[var].isnull().sum().values)}')
raw_sliced = running_window_slice(raw, date)
logger.info(f'NaNs in raw running window: {float(raw_sliced[var].isnull().sum().values)}')
logger.info('Datasets sliced into running windows.')

# Create CDFs from sliced datasets
obs_cdf = ECDF(obs_sliced[var].values.flatten())
hist_cdf = ECDF(hist_sliced[var].values.flatten())
raw_cdf = ECDF(raw_sliced[var].values.flatten())

# Select day of interest and create mask
day = int(date.strftime('%d'))
month = int(date.strftime('%m'))
mask = (raw.time.dt.month == month) & (raw.time.dt.day == day)

# Apply mask to dataset
raw_date = raw.where(mask, drop = True) # This should select day of interest across all years in ds
logger.info(f'NaNs in raw data for {date}: {float(raw_date[var].isnull().sum().values)}')
logger.info(f'Dataset sliced down to date of interest: {date}')
logger.info(raw_date)

# Use raw_cdf like a function to find all of the percentiles in raw_date
raw_percentiles = raw_cdf(raw_date[var].values)
logger.info(f'Percentile values for raw data: {raw_percentiles}')
logger.info(f'NaNs in raw_percentiles: {float(np.isnan(raw_percentiles).sum())}')

# TODO: NaNs are appearing in obs and hist from this point on
# Invert hist_cdf and obs_cdf to take a percentile and output a data point
inv_obs = invert_ecdf(obs_cdf)
inv_hist = invert_ecdf(hist_cdf)

# Find data points for every percentile in raw_percentiles
obs_vals = inv_obs(raw_percentiles)
logger.info(f'NaNs in obs_vals: {float(np.isnan(obs_vals).sum())}')
logger.info(f'Obs values for given percentiles: {obs_vals}')
hist_vals = inv_hist(raw_percentiles)

14:30:22 |     INFO | plots.validation:loc_sel:line 89 - Selecting nearest location to 40.788, -111.978.
/uufs/chpc.utah.edu/common/home/strong-group7/sydney/miniforge3_envs/olympics/lib/python3.11/site-packages/xarray/core/dataarray.py:6318: FutureWarning: Behaviour of argmin/argmax with neither dim nor axis argument will change to return a dict of indices of each dimension. To get a single, flat index, please use np.argmin(da.data) or np.argmax(da.data) instead of da.argmin() or da.argmax().
  result = self.variable.argmin(dim, axis, keep_attrs, skipna)
14:30:22 |     INFO | plots.validation:loc_sel:line 100 - Point selected is <xarray.DataArray ()> Size: 8B
array(3916)
Coordinates:
    time     datetime64[ns] 8B 1985-01-01 from 40.788, -111.978.
14:30:22 |     INFO | plots.validation:loc_sel:line 102 - Min distance is located at 46, 43.
14:30:22 |     INFO | plots.validation:loc_sel:line 113 - New lat: 40.78413391113281
14:30:22 |     INFO | plots.validation:loc_sel:line 114 - New l

In [ ]:
inv_obs

In [ ]:
type(inv_obs)

scipy.interpolate._interpolate.interp1d

In [ ]:
obs_vals

array([[[274.31171836, 275.26440219, 277.04324364, ..., 276.83082211,
         276.21550865, 275.25505373],
        [274.27933781, 274.78148161, 276.75992212, ..., 276.43825293,
         275.4444627 , 274.95730403],
        [274.25275933, 274.80238722, 276.75643018, ..., 275.64273315,
         275.48474395, 274.68663898],
        ...,
        [274.48789747, 274.36266929, 274.41390424, ..., 270.93589327,
         270.82304639, 270.8538696 ],
        [274.24593738, 274.24416982, 274.12847408, ..., 270.5828912 ,
         270.44838994, 270.36918429],
        [273.98614999, 274.02516669, 273.99448461, ..., 270.51538629,
         270.26133418, 269.86278954]],

       [[275.62425927, 277.23918422, 278.55070337, ..., 277.5435586 ,
         276.88880034, 276.05152562],
        [274.9093484 , 275.32827368, 277.91559556, ..., 276.91774389,
         275.71460491, 275.68065641],
        [274.55419137, 275.31342678, 277.76023697, ..., 275.7924203 ,
         275.35729985, 275.26554642],
        ...,


In [ ]:
inv_obs(0.85)

array(278.53671949)

In [ ]:
raw_percentiles

array([[[0.48731337, 0.58359361, 0.74885076, ..., 0.73150837,
         0.67745451, 0.58266718],
        [0.48395035, 0.53425409, 0.72547058, ..., 0.69762175,
         0.60205316, 0.55219242],
        [0.48140227, 0.53654153, 0.7251805 , ..., 0.62200352,
         0.6060242 , 0.52465321],
        ...,
        [0.50464265, 0.49222207, 0.49744249, ..., 0.20890371,
         0.20210791, 0.20391424],
        [0.48075919, 0.48058999, 0.46918866, ..., 0.18804389,
         0.18052235, 0.17619774],
        [0.4551072 , 0.45907136, 0.45594194, ..., 0.18428939,
         0.17040406, 0.15032926]],

       [[0.62020239, 0.76426101, 0.85075103, ..., 0.78690076,
         0.73620013, 0.66189749],
        [0.54742849, 0.5902751 , 0.81267868, ..., 0.73871437,
         0.62922763, 0.62568622],
        [0.51135183, 0.58877368, 0.8021158 , ..., 0.63674581,
         0.59315503, 0.58370601],
        ...,
        [0.51575828, 0.51045727, 0.51162281, ..., 0.17164143,
         0.1544435 , 0.13494984],
        [0.5

In [ ]:
inv_obs(0.26744119)

array(271.81312427)

In [ ]:
np.isnan(obs_vals)

array([[[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, Fal

In [ ]:
np.isnan(obs_vals).sum()

np.int64(0)

In [ ]:
np.isnan(hist_vals).sum()

np.int64(0)

In [ ]:
"""
Author: Sydney Smith
Date Created: September 29, 2026
"""

from datetime import datetime
import gc
import glob
from loguru import logger
import numpy as np
import os
import pandas as pd
from pathlib import Path
from scipy.interpolate import interp1d
from statsmodels.distributions.empirical_distribution import ECDF, monotone_fn_inverter
import sys
import time
import xarray as xr

# ==================================
# - Establish Relative File Path - 
# ==================================

current_dir = Path(__file__).resolve().parent
sys.path.append(str(current_dir))

from plots.validation import running_window_slice, loc_sel

# ===================
# - Set Up Logger - 
# ==================

# Create directory for log files if it doesn't already exist
log_path = str(current_dir / 'log')
os.makedirs(log_path, exist_ok = True)

# String filename
log_filename = datetime.now().strftime('log/%Y-%m-%d_%H-%M-%S.log')

# Custom format for log prints
log_format = log_format = '<cyan>{time:HH:mm:ss}</cyan> | <level>{level:>8}</level> | <yellow>{name}</yellow>:<cyan>{function}</cyan>:line <magenta>{line}</magenta> - <level>{message}</level>'

# Removes the default stderr sink
logger.remove()  
# Anything above info to console
logger.add(sys.stderr, colorize = True, format = log_format, level = 'INFO') 
# Anything above debug to log file
logger.add(log_filename, colorize = False, format = log_format, level = 'DEBUG')
logger.debug(f'Log files saved to {log_path}')

# =====================
# ---- Functions ----
# =====================

def invert_ecdf(ecdf_obj):
    """
    Inverts a statsmodels ECDF object using its .y and .x attributes, regardless of if
    the values are monotonic or not, with safe extrapolation to prevent NaNs.
    """
    return interp1d(
        ecdf_obj.y, # Pass y values as x values
        ecdf_obj.x, # And vice versa
        bounds_error = False, 
        fill_value = 'extrapolate'  # Prevents NaNs if raw percentiles push past bounds
    )

def apply_to_date(obs: xr.Dataset, hist, raw, var, date) -> np.ndarray:
    """
    Debiases data for a given date by constructing a CDF using a dataset restricted to be within a 
    running window. Date must be a valid date that exists within the dataset formatted as '%Y-%m-%d' 
    datetime object.
    """
    # TODO: check how running_window_slice handles leap years

    # Slice data into 31 day running windows centered on given date (note: 31 days is defualt setting)
    obs_sliced = running_window_slice(obs, date)
    logger.info(f'NaNs in obs running window: {float(obs_sliced[var].isnull().sum().values)}')
    hist_sliced = running_window_slice(hist, date)
    logger.info(f'NaNs in hist running window: {float(hist_sliced[var].isnull().sum().values)}')
    raw_sliced = running_window_slice(raw, date)
    logger.info(f'NaNs in raw running window: {float(raw_sliced[var].isnull().sum().values)}')
    logger.info('Datasets sliced into running windows.')

    # Create CDFs from sliced datasets
    obs_cdf = ECDF(obs_sliced[var].values.flatten())
    hist_cdf = ECDF(hist_sliced[var].values.flatten())
    raw_cdf = ECDF(raw_sliced[var].values.flatten())

    # Select day of interest and create mask
    day = int(date.strftime('%d'))
    month = int(date.strftime('%m'))
    mask = (raw.time.dt.month == month) & (raw.time.dt.day == day)

    # Skip dates that don't exist
    if not mask.any():
        logger.warning(f'No date found for {date}.')
        return None
    
    # Apply mask to dataset
    raw_date = raw.where(mask, drop = True) # This should select day of interest across all years in ds
    logger.info(f'NaNs in raw data for {date}: {float(raw_date[var].isnull().sum().values)}')
    logger.info(f'Dataset sliced down to date of interest: {date}')
    logger.info(raw_date)

    # Use raw_cdf like a function to find all of the percentiles in raw_date
    raw_percentiles = raw_cdf(raw_date[var].values)
    logger.info(f'Percentile values for raw data: {raw_percentiles}')
    logger.info(f'NaNs in raw_percentiles: {float(np.isnan(raw_percentiles).sum())}')

    # TODO: NaNs are appearing in obs and hist from this point on
    # Invert hist_cdf and obs_cdf to take a percentile and output a data point
    inv_obs = invert_ecdf(obs_cdf)
    inv_hist = invert_ecdf(hist_cdf)

    # Find data points for every percentile in raw_percentiles
    obs_vals = inv_obs(raw_percentiles)
    logger.info(f'NaNs in obs_vals: {float(np.isnan(obs_vals).sum())}')
    logger.info(f'Obs values for given percentiles: {obs_vals}')
    hist_vals = inv_hist(raw_percentiles)
    logger.info(f'NaNs in hist_vals: {float(np.isnan(hist_vals).sum())}')
    logger.info(f'Historical values for given percentiles: {hist_vals}')

    # Perform bias adjustment
    bias = hist_vals - obs_vals
    logger.info(f'NaNs in bias: {float(np.isnan(bias).sum())}')
    raw_bias_corrected = raw_date[var].values - bias

    # End script early if there are Nans in data
    if np.isnan(raw_bias_corrected).any():
        logger.error(f'NaNs in debiased data: {float(np.isnan(raw_bias_corrected).sum())}')
        logger.info(f'Nan error occured on {date}.')
        return None

    logger.info('Raw data successfully bias corrected!')
    logger.info(raw_bias_corrected)
    logger.info(f'Bias corrected max: {raw_bias_corrected.max()}')
    logger.info(f'Bias corrected min: {raw_bias_corrected.min()}')

    # Close out of unwanted datasets
    obs_sliced.close()
    hist_sliced.close()
    raw_sliced.close()
    raw_date.close()
    del obs_vals
    del hist_vals
    del raw_percentiles
    gc.collect()

    return raw_bias_corrected

def ECDFM(obs, raw, var, lat, lon):
    """
    Performs ECDFM quantile mapping debiasing process on given data based on data from a given histroical
    period and corresponding observational data. Raw data should include both historical and future data. 
    """

    # Slice raw data down to historical period
    hist = raw.sel(time = slice('1985-01-01', '2014-12-31'))
    
    # Select specific location of interest (choses nearest possible location)
    obs_loc = loc_sel(obs, lat = lat, lon = lon)
    logger.info(f'NaNs at selected obs location: {float(obs_loc[var].isnull().sum().values)}')
    hist_loc = loc_sel(hist, lat = lat, lon = lon)
    logger.info(f'NaNs at selected hist location: {float(hist_loc[var].isnull().sum().values)}')
    raw_loc = loc_sel(raw, lat = lat, lon = lon)
    logger.info(f'NaNs at selected raw location: {float(raw_loc[var].isnull().sum().values)}')
    logger.info(raw_loc[var])

    # Make copy of raw_loc to plug debiased data into
    template = raw_loc.copy()

    # Create date range that spans a calendar year
    dates = pd.date_range(start = '1985-01-01', end = '1985-12-31', freq = 'D') 

    for date in dates:
        # Apply bias correction for single date
        logger.info(f'Date type: {type(date)}')
        debiased_doy = apply_to_date(obs_loc, hist_loc, raw_loc, var, date)

        # Exit out of function if Nans occur
        if debiased_doy is None:
            logger.error(f'Nans detected in dataset. See log above for more information.')
            return None

        # Pull month and day from date
        day = int(date.strftime('%d'))
        month = int(date.strftime('%m'))

        # Plug in debiased data
        mask = (template.time.dt.month == month) & (template.time.dt.day == day)
        template[var].data[mask] = debiased_doy 
        logger.info(f'Debiased data for {date} added to dataset!')

        # Check that data was actaully replaced
        temp_masked = template[var].where(mask, drop = True)
        raw_masked = raw[var].where(mask, drop = True)
        bias = raw_masked - temp_masked # Back into data's bias
        if (bias.all() == 0): # Ensure it's not all zeros
            logger.error('Bias corrected data was not successfully applied to dataset.')
            return None
        else:
            logger.info(f'Bias for {date} successfully corrected.')

        # Close unnecessay datasets
        temp_masked.close()
        raw_masked.close()

    return template

def nan_test(var):
    """
    Returns location of Nans in debiased data.
    """
    
    fpath = glob.glob(str(current_dir / 'wrfout' / f'*{var}*.nc'))
    debiased = xr.open_dataset(fpath[0], decode_times = True)

    nan_data = debiased[var].where(debiased[var].isnull(), drop = True)

    return nan_data

# Catch silent errors and report to log file
@logger.catch 
def main(var, data_location, save = True):
    # Open datasets
    obs_path = glob.glob(str(data_location / 'gridMET' / f'*{var}*.nc'))
    obs = xr.open_dataset(obs_path[0], decode_times = True)

    raw_path = glob.glob(str(data_location / 'daily' / f'*{var}*.nc'))
    raw = xr.open_dataset(raw_path[0], decode_times = True)

    # Set location of interest as SLC airport
    lat = 40.788
    lon = -111.978

    # Pass datasets to debiaser
    test = ECDFM(obs, raw, var, lat, lon)

    if save:
        # Create output directory to store new cleaned files
        output_dir = current_dir / 'wrfout' 
        os.makedirs(output_dir, exist_ok = True) # Don't make if it already exists

        # Generate save name for debiased data
        out_path = os.path.join(output_dir, f'wrfout_GSLBIP_multimodel_ssp245_{var}_test.nc')

        # Save data to netCDF file
        test.to_netcdf(out_path)
        logger.success(f'File saved to: {out_path}')

    # Close out of data
    test.close()

# if __name__ == '__main__':

#     # Track program time in log files
#     start = time.perf_counter()
#     logger.info('Beginning execution.')

#     # Only inputs required
#     main(
#             var = 'tmmx',
#             data_location = current_dir,
#             save = False
#             )

#     # Report of runtime at completion 
#     logger.success(f'Debiasing process completed!')
#     logger.info(f'Total runtime: {time.perf_counter() - start:.4f}s')

#     # Force script to stop running once code is finished
#     sys.exit(0)

var = 'tmmx'
# Open datasets
obs_path = glob.glob(str(current_dir / 'gridMET' / f'*{var}*.nc'))
obs = xr.open_dataset(obs_path[0], decode_times = True)

raw_path = glob.glob(str(current_dir / 'daily' / f'*{var}*.nc'))
raw = xr.open_dataset(raw_path[0], decode_times = True)

# Set location of interest as SLC airport
lat = 40.788
lon = -111.978

# Slice raw data down to historical period
hist = raw.sel(time = slice('1985-01-01', '2014-12-31'))

# Select specific location of interest (choses nearest possible location)
obs_loc = loc_sel(obs, lat = lat, lon = lon)
logger.info(f'NaNs at selected obs location: {float(obs_loc[var].isnull().sum().values)}')
hist_loc = loc_sel(hist, lat = lat, lon = lon)
logger.info(f'NaNs at selected hist location: {float(hist_loc[var].isnull().sum().values)}')
raw_loc = loc_sel(raw, lat = lat, lon = lon)
logger.info(f'NaNs at selected raw location: {float(raw_loc[var].isnull().sum().values)}')
logger.info(raw_loc[var])

# Make copy of raw_loc to plug debiased data into
template = raw_loc.copy()

# Create date range that spans a calendar year
date = pd.to_datetime('2013-11-02') 

# Slice data into 31 day running windows centered on given date (note: 31 days is defualt setting)
obs_sliced = running_window_slice(obs, date)
logger.info(f'NaNs in obs running window: {float(obs_sliced[var].isnull().sum().values)}')
hist_sliced = running_window_slice(hist, date)
logger.info(f'NaNs in hist running window: {float(hist_sliced[var].isnull().sum().values)}')
raw_sliced = running_window_slice(raw, date)
logger.info(f'NaNs in raw running window: {float(raw_sliced[var].isnull().sum().values)}')
logger.info('Datasets sliced into running windows.')

# Create CDFs from sliced datasets
obs_cdf = ECDF(obs_sliced[var].values.flatten())
hist_cdf = ECDF(hist_sliced[var].values.flatten())
raw_cdf = ECDF(raw_sliced[var].values.flatten())

# Select day of interest and create mask
day = int(date.strftime('%d'))
month = int(date.strftime('%m'))
mask = (raw.time.dt.month == month) & (raw.time.dt.day == day)

# Apply mask to dataset
raw_date = raw.where(mask, drop = True) # This should select day of interest across all years in ds
logger.info(f'NaNs in raw data for {date}: {float(raw_date[var].isnull().sum().values)}')
logger.info(f'Dataset sliced down to date of interest: {date}')
logger.info(raw_date)

# Use raw_cdf like a function to find all of the percentiles in raw_date
raw_percentiles = raw_cdf(raw_date[var].values)
logger.info(f'Percentile values for raw data: {raw_percentiles}')
logger.info(f'NaNs in raw_percentiles: {float(np.isnan(raw_percentiles).sum())}')

# TODO: NaNs are appearing in obs and hist from this point on
# Invert hist_cdf and obs_cdf to take a percentile and output a data point
inv_obs = invert_ecdf(obs_cdf)
inv_hist = invert_ecdf(hist_cdf)

# Find data points for every percentile in raw_percentiles
obs_vals = inv_obs(raw_percentiles)
logger.info(f'NaNs in obs_vals: {float(np.isnan(obs_vals).sum())}')
logger.info(f'Obs values for given percentiles: {obs_vals}')
hist_vals = inv_hist(raw_percentiles)

14:39:13 |     INFO | plots.validation:loc_sel:line 89 - Selecting nearest location to 40.788, -111.978.
/uufs/chpc.utah.edu/common/home/strong-group7/sydney/miniforge3_envs/olympics/lib/python3.11/site-packages/xarray/core/dataarray.py:6318: FutureWarning: Behaviour of argmin/argmax with neither dim nor axis argument will change to return a dict of indices of each dimension. To get a single, flat index, please use np.argmin(da.data) or np.argmax(da.data) instead of da.argmin() or da.argmax().
  result = self.variable.argmin(dim, axis, keep_attrs, skipna)
14:39:13 |     INFO | plots.validation:loc_sel:line 100 - Point selected is <xarray.DataArray ()> Size: 8B
array(3916)
Coordinates:
    time     datetime64[ns] 8B 1985-01-01 from 40.788, -111.978.
14:39:13 |     INFO | plots.validation:loc_sel:line 102 - Min distance is located at 46, 43.
14:39:13 |     INFO | plots.validation:loc_sel:line 113 - New lat: 40.78413391113281
14:39:13 |     INFO | plots.validation:loc_sel:line 114 - New l

In [ ]:
np.isnan(hist_vals).sum()

np.int64(3)

In [ ]:
hist_vals.where(np.isnan(hist_vals), drop = True)

AttributeError: 'numpy.ndarray' object has no attribute 'where'

In [ ]:
hist_vals

array([[[283.16326904, 284.96707153, 286.39117432, ..., 283.89645386,
         283.66320801, 282.86773682],
        [282.68914795, 285.56109619, 287.31497192, ..., 284.15637207,
         283.84881592, 282.35986328],
        [282.97567749, 285.55603027, 287.36486816, ..., 283.56747437,
         283.46044922, 281.9078064 ],
        ...,
        [283.17053223, 283.69366455, 283.66918945, ..., 269.62347412,
         269.4671936 , 268.21018916],
        [282.92816162, 283.45425415, 283.41189575, ..., 269.12808228,
         268.8238459 , 267.9191178 ],
        [282.6745287 , 282.7789917 , 282.84579468, ..., 268.4927063 ,
         268.07351419, 267.24388189]],

       [[275.03088379, 276.93804932, 278.72470093, ..., 281.48013306,
         280.18310547, 279.17791748],
        [274.44683838, 277.53497314, 279.32931519, ..., 282.16680908,
         281.75439453, 279.71905518],
        [275.27661133, 277.49505615, 279.33422852, ..., 281.63293457,
         281.47271729, 278.45324707],
        ...,


In [ ]:
nan_only = hist_vals[np.isnan(hist_vals)]

In [ ]:
nan_only

array([nan, nan, nan])

In [ ]:
np.isnan(hist_vals).sum()

np.int64(3)

In [ ]:
np.isnan(raw_percentiles).sum()

np.int64(0)

In [ ]:
inv_obs

In [ ]:
np.where(np.isnan(obs_vals))

(array([28, 28, 28]), array([44, 45, 45]), array([88, 87, 88]))

In [ ]:
obs_vals[28, 28, 28]

np.float64(269.97555057832506)

In [ ]:
inv_obs(269.97555057832506)

array(3.22707674e+08)

In [ ]:
obs_vals[44, 45, 45]

np.float64(285.5702384986018)

In [ ]:
inv_obs(285.5702384986018)

array(3.4141763e+08)

In [ ]:
obs_vals[88, 87, 88]

np.float64(278.18308339552345)

In [ ]:
inv_obs(278.18308339552345)

array(3.32554782e+08)

In [ ]:
inv_obs(278)

array(3.32335125e+08)

In [ ]:
inv_obs(280)

array(3.34734654e+08)

In [ ]:
inv_obs(285)

array(3.40733478e+08)

In [ ]:
inv_obs(273)

array(3.26336301e+08)

In [ ]:
obs_vals

array([[[285.64519137, 287.4341393 , 288.69169425, ..., 286.39666091,
         286.1595603 , 285.34910981],
        [285.17091289, 287.97876065, 289.46858277, ..., 286.6550964 ,
         286.34823944, 284.83822725],
        [285.45675915, 287.97426292, 289.50920185, ..., 286.06138513,
         285.95050428, 284.3872099 ],
        ...,
        [285.65281134, 286.19136937, 286.16585138, ..., 273.09787797,
         272.98626955, 272.09945705],
        [285.40823675, 285.94396539, 285.9       , ..., 272.74291801,
         272.52866346, 271.89951695],
        [285.15601433, 285.26074616, 285.32686828, ..., 272.29640219,
         272.00622357, 271.43366865]],

       [[277.90805467, 279.66475354, 281.36334716, ..., 283.96525297,
         282.73789036, 281.79142911],
        [277.35552916, 280.22306165, 281.93347158, ..., 284.64391662,
         284.23448066, 282.3       ],
        [278.13446015, 280.18723346, 281.93821428, ..., 284.11471392,
         283.95765235, 281.10128873],
        ...,


In [ ]:
raw_percentiles[28, 28, 28]

np.float64(0.016214166480646143)

In [ ]:
inv_obs(0.016214166480646143)

array(269.97555058)

In [ ]:
shape(obs_vals)

NameError: name 'shape' is not defined

In [ ]:
np.shape(obs_vals)

(115, 111, 90)

In [ ]:
np.shape(raw_percentiles)

(115, 111, 90)

In [ ]:
raw_percentiles[44, 45, 45]

np.float64(0.6039288657521477)

In [ ]:
raw_percentiles[88,87,88]

np.float64(0.20827683223195145)

In [ ]:
inv_obs(0.20827683223195145)

array(278.1830834)

In [ ]:
inv_obs(0.6039288657521477)

array(285.5702385)

In [ ]:
inv_obs(raw_percentiles)

/uufs/chpc.utah.edu/common/home/strong-group7/sydney/miniforge3_envs/olympics/lib/python3.11/site-packages/scipy/interpolate/_interpolate.py:504: RuntimeWarning: invalid value encountered in add
  y_new = slope*(x_new - x_lo)[:, None] + y_lo


array([[[285.64519137, 287.4341393 , 288.69169425, ..., 286.39666091,
         286.1595603 , 285.34910981],
        [285.17091289, 287.97876065, 289.46858277, ..., 286.6550964 ,
         286.34823944, 284.83822725],
        [285.45675915, 287.97426292, 289.50920185, ..., 286.06138513,
         285.95050428, 284.3872099 ],
        ...,
        [285.65281134, 286.19136937, 286.16585138, ..., 273.09787797,
         272.98626955, 272.09945705],
        [285.40823675, 285.94396539, 285.9       , ..., 272.74291801,
         272.52866346, 271.89951695],
        [285.15601433, 285.26074616, 285.32686828, ..., 272.29640219,
         272.00622357, 271.43366865]],

       [[277.90805467, 279.66475354, 281.36334716, ..., 283.96525297,
         282.73789036, 281.79142911],
        [277.35552916, 280.22306165, 281.93347158, ..., 284.64391662,
         284.23448066, 282.3       ],
        [278.13446015, 280.18723346, 281.93821428, ..., 284.11471392,
         283.95765235, 281.10128873],
        ...,


In [ ]:
test = inv_obs(raw_percentiles)

/uufs/chpc.utah.edu/common/home/strong-group7/sydney/miniforge3_envs/olympics/lib/python3.11/site-packages/scipy/interpolate/_interpolate.py:504: RuntimeWarning: invalid value encountered in add
  y_new = slope*(x_new - x_lo)[:, None] + y_lo


In [ ]:
np.isnan(test).sum()

np.int64(3)

In [ ]:
np.where(np.isnan(test))

(array([28, 28, 28]), array([44, 45, 45]), array([88, 87, 88]))

In [ ]:
obs_vals[28, 28, 28]

np.float64(269.97555057832506)

In [ ]:
test[28, 28, 28]

np.float64(269.97555057832506)

In [ ]:
test[44, 45, 45]

np.float64(285.5702384986018)

In [ ]:
test[28,44,88]

np.float64(nan)

In [ ]:
raw_percentiles[28,44,88]

np.float64(2.8078569453043507e-08)

In [ ]:
np.argwhere(np.isnan(hist_vals))

array([[28, 44, 88],
       [28, 45, 87],
       [28, 45, 88]])

In [ ]:
raw_percentiles[28, 45, 87]

np.float64(8.423570835913052e-08)

In [ ]:
raw_percentiles[28, 45, 88]

np.float64(5.6157138906087014e-08)

In [ ]:
in_obs(5.6157138906087014e-08)

NameError: name 'in_obs' is not defined

In [ ]:
inv_obs(5.6157138906087014e-08)

/uufs/chpc.utah.edu/common/home/strong-group7/sydney/miniforge3_envs/olympics/lib/python3.11/site-packages/scipy/interpolate/_interpolate.py:504: RuntimeWarning: invalid value encountered in add
  y_new = slope*(x_new - x_lo)[:, None] + y_lo


array(nan)

In [ ]:
inv_obs(0.01)

array(268.97482178)

In [ ]:
inv_obs(0.001)

array(265.18849577)

In [ ]:
inv_obs(0.0001)

array(261.47479356)

In [ ]:
inv_obs(0.0000000000001)

/uufs/chpc.utah.edu/common/home/strong-group7/sydney/miniforge3_envs/olympics/lib/python3.11/site-packages/scipy/interpolate/_interpolate.py:504: RuntimeWarning: invalid value encountered in add
  y_new = slope*(x_new - x_lo)[:, None] + y_lo


array(nan)

In [ ]:
inv_obs(0.00001)

array(258.32017371)

In [ ]:
inv_obs(0.000001)

array(256.92458736)

In [ ]:
inv_obs(0.0000001)

array(nan)

In [ ]:
inv_obs(0.0000009)

array(256.90490296)

In [ ]:
inv_obs(0.0000002)

array(256.5693837)

In [ ]:
inv_obs(0.00000011)

array(256.52562662)

In [ ]:
inv_obs(0.000000100000001)

array(nan)

In [ ]:
inv_obs(0.00000100000001)

array(256.92458737)

In [ ]:
y_vals = obs_cdf.y
x_vals = obs_cdf.x
    
# interp1d strictly requires the first argument (x) to be unique and strictly increasing.
# np.unique automatically sorts the array and returns unique elements along with their indices.
y_unique, indices = np.unique(y_vals, return_index=True)
x_unique = x_vals[indices]

In [ ]:
y_unique

array([0.00000000e+00, 1.07634516e-07, 2.15269032e-07, ...,
       9.99999785e-01, 9.99999892e-01, 1.00000000e+00], shape=(9290701,))

In [ ]:
y_vals

array([0.00000000e+00, 1.07634516e-07, 2.15269032e-07, ...,
       9.99999785e-01, 9.99999892e-01, 1.00000000e+00], shape=(9290701,))

In [ ]:
y_vals = obs_cdf.y
ysort = np.sort(y_vals)
x_vals = obs_cdf.x
    
# interp1d strictly requires the first argument (x) to be unique and strictly increasing.
# np.unique automatically sorts the array and returns unique elements along with their indices.
y_unique, indices = np.unique(y_vals, return_index=True)
x_unique = x_vals[indices]

In [ ]:
ysort

array([0.00000000e+00, 1.07634516e-07, 2.15269032e-07, ...,
       9.99999785e-01, 9.99999892e-01, 1.00000000e+00], shape=(9290701,))

In [ ]:
ysort-y_unique

array([0., 0., 0., ..., 0., 0., 0.], shape=(9290701,))

In [ ]:
y_vals = obs_cdf.y
ysort = np.sort(y_vals)
diff=ysort-y_vals
x_vals = obs_cdf.x
    
# interp1d strictly requires the first argument (x) to be unique and strictly increasing.
# np.unique automatically sorts the array and returns unique elements along with their indices.
y_unique, indices = np.unique(y_vals, return_index=True)
x_unique = x_vals[indices]

In [ ]:
diff

array([0., 0., 0., ..., 0., 0., 0.], shape=(9290701,))

In [ ]:
np.where(diff!=0)

(array([], dtype=int64),)

In [ ]:
x_unique

array([        -inf, 256.52447655, 256.57680735, ..., 303.14376005,
       303.32639569, 303.45553179], shape=(9290701,))

In [ ]:
indicies

NameError: name 'indicies' is not defined

In [ ]:
indices

array([      0,       1,       2, ..., 9290698, 9290699, 9290700],
      shape=(9290701,))

In [ ]:
x_vals

array([        -inf, 256.52447655, 256.57680735, ..., 303.14376005,
       303.32639569, 303.45553179], shape=(9290701,))

In [ ]:
obs_cdf

In [ ]:
obs_cdf.x

array([        -inf, 256.52447655, 256.57680735, ..., 303.14376005,
       303.32639569, 303.45553179], shape=(9290701,))

In [ ]:
np.where(obs_cdf.x==inf)

NameError: name 'inf' is not defined

In [ ]:
"""
Author: Sydney Smith
Date Created: September 29, 2026
"""

from datetime import datetime
import gc
import glob
from loguru import logger
import numpy as np
import os
import pandas as pd
from pathlib import Path
from scipy.interpolate import interp1d
from statsmodels.distributions.empirical_distribution import ECDF, monotone_fn_inverter
import sys
import time
import xarray as xr

# ==================================
# - Establish Relative File Path - 
# ==================================

current_dir = Path(__file__).resolve().parent
sys.path.append(str(current_dir))

from plots.validation import running_window_slice, loc_sel

# ===================
# - Set Up Logger - 
# ==================

# Create directory for log files if it doesn't already exist
log_path = str(current_dir / 'log')
os.makedirs(log_path, exist_ok = True)

# String filename
log_filename = datetime.now().strftime('log/%Y-%m-%d_%H-%M-%S.log')

# Custom format for log prints
log_format = log_format = '<cyan>{time:HH:mm:ss}</cyan> | <level>{level:>8}</level> | <yellow>{name}</yellow>:<cyan>{function}</cyan>:line <magenta>{line}</magenta> - <level>{message}</level>'

# Removes the default stderr sink
logger.remove()  
# Anything above info to console
logger.add(sys.stderr, colorize = True, format = log_format, level = 'INFO') 
# Anything above debug to log file
logger.add(log_filename, colorize = False, format = log_format, level = 'DEBUG')
logger.debug(f'Log files saved to {log_path}')

# =====================
# ---- Functions ----
# =====================

def invert_ecdf(ecdf_obj):
    """
    Inverts a statsmodels ECDF object using its .y and .x attributes, regardless of if
    the values are monotonic or not, with safe extrapolation to prevent NaNs.
    """
    # Pull out any indefinite values
    finite_mask = np.isfinite(ecdf_obj.x)
    x_vals = ecdf_obj.x[finite_mask]
    y_vals = ecdf_obj.y[finite_mask]

    # Ensure x_vals are unique and strictly increasing for interp1d
    x_unique, indices = np.unique(x_vals, return_index=True)
    y_unique = y_vals[indices]

    return interp1d(
        y_unique, # Pass y values as x values
        x_unique, # And vice versa
        bounds_error = False, 
        fill_value = 'extrapolate'  # Prevents NaNs if raw percentiles push past bounds
    )

def apply_to_date(obs: xr.Dataset, hist, raw, var, date) -> np.ndarray:
    """
    Debiases data for a given date by constructing a CDF using a dataset restricted to be within a 
    running window. Date must be a valid date that exists within the dataset formatted as '%Y-%m-%d' 
    datetime object.
    """
    # TODO: check how running_window_slice handles leap years

    # Slice data into 31 day running windows centered on given date (note: 31 days is defualt setting)
    obs_sliced = running_window_slice(obs, date)
    logger.info(f'NaNs in obs running window: {float(obs_sliced[var].isnull().sum().values)}')
    hist_sliced = running_window_slice(hist, date)
    logger.info(f'NaNs in hist running window: {float(hist_sliced[var].isnull().sum().values)}')
    raw_sliced = running_window_slice(raw, date)
    logger.info(f'NaNs in raw running window: {float(raw_sliced[var].isnull().sum().values)}')
    logger.info('Datasets sliced into running windows.')

    # End function if NaNs are present in the source data
    if obs_sliced[var].isnull().any() or hist_sliced[var].isnull().any() or raw_sliced[var].isnull().any():
        logger.error('NaNs detected in source data.')
        return NotImplementedError

    # Create CDFs from sliced datasets
    obs_cdf = ECDF(obs_sliced[var].values.flatten())
    hist_cdf = ECDF(hist_sliced[var].values.flatten())
    raw_cdf = ECDF(raw_sliced[var].values.flatten())

    # Select day of interest and create mask
    day = int(date.strftime('%d'))
    month = int(date.strftime('%m'))
    mask = (raw.time.dt.month == month) & (raw.time.dt.day == day)

    # Skip dates that don't exist
    if not mask.any():
        logger.warning(f'No date found for {date}.')
        return None
    
    # Apply mask to dataset
    raw_date = raw.where(mask, drop = True) # This should select day of interest across all years in ds
    logger.info(f'NaNs in raw data for {date}: {float(raw_date[var].isnull().sum().values)}')
    logger.info(f'Dataset sliced down to date of interest: {date}')
    logger.info(raw_date)

    # Use raw_cdf like a function to find all of the percentiles in raw_date
    raw_percentiles = raw_cdf(raw_date[var].values)
    logger.info(f'Percentile values for raw data: {raw_percentiles}')
    logger.info(f'NaNs in raw_percentiles: {float(np.isnan(raw_percentiles).sum())}')

    # TODO: NaNs are appearing in obs and hist from this point on
    # Invert hist_cdf and obs_cdf to take a percentile and output a data point
    inv_obs = invert_ecdf(obs_cdf)
    inv_hist = invert_ecdf(hist_cdf)

    # Find data points for every percentile in raw_percentiles
    obs_vals = inv_obs(raw_percentiles)
    logger.info(f'NaNs in obs_vals: {float(np.isnan(obs_vals).sum())}')
    logger.info(f'Obs values for given percentiles: {obs_vals}')
    hist_vals = inv_hist(raw_percentiles)
    logger.info(f'NaNs in hist_vals: {float(np.isnan(hist_vals).sum())}')
    logger.info(f'Historical values for given percentiles: {hist_vals}')

    # Perform bias adjustment
    bias = hist_vals - obs_vals
    logger.info(f'NaNs in bias: {float(np.isnan(bias).sum())}')
    raw_bias_corrected = raw_date[var].values - bias

    # End script early if there are Nans in data
    if np.isnan(raw_bias_corrected).any():
        logger.error(f'NaNs in debiased data: {float(np.isnan(raw_bias_corrected).sum())}')
        logger.info(f'Nan error occured on {date}.')
        return None

    logger.info('Raw data successfully bias corrected!')
    logger.info(raw_bias_corrected)
    logger.info(f'Bias corrected max: {raw_bias_corrected.max()}')
    logger.info(f'Bias corrected min: {raw_bias_corrected.min()}')

    # Close out of unwanted datasets
    obs_sliced.close()
    hist_sliced.close()
    raw_sliced.close()
    raw_date.close()
    del obs_vals
    del hist_vals
    del raw_percentiles
    gc.collect()

    return raw_bias_corrected

def ECDFM(obs, raw, var, lat, lon):
    """
    Performs ECDFM quantile mapping debiasing process on given data based on data from a given histroical
    period and corresponding observational data. Raw data should include both historical and future data. 
    """

    # Slice raw data down to historical period
    hist = raw.sel(time = slice('1985-01-01', '2014-12-31'))
    
    # Select specific location of interest (choses nearest possible location)
    obs_loc = loc_sel(obs, lat = lat, lon = lon)
    logger.info(f'NaNs at selected obs location: {float(obs_loc[var].isnull().sum().values)}')
    hist_loc = loc_sel(hist, lat = lat, lon = lon)
    logger.info(f'NaNs at selected hist location: {float(hist_loc[var].isnull().sum().values)}')
    raw_loc = loc_sel(raw, lat = lat, lon = lon)
    logger.info(f'NaNs at selected raw location: {float(raw_loc[var].isnull().sum().values)}')
    logger.info(raw_loc[var])

    # Make copy of raw_loc to plug debiased data into
    template = raw_loc.copy()

    # Create date range that spans a calendar year
    dates = pd.date_range(start = '1985-01-01', end = '1985-12-31', freq = 'D') 

    for date in dates:
        # Apply bias correction for single date
        logger.info(f'Date type: {type(date)}')
        debiased_doy = apply_to_date(obs_loc, hist_loc, raw_loc, var, date)

        # Exit out of function if Nans occur
        if debiased_doy is None:
            logger.error(f'Nans detected in dataset. See log above for more information.')
            return None

        # Pull month and day from date
        day = int(date.strftime('%d'))
        month = int(date.strftime('%m'))

        # Plug in debiased data
        mask = (template.time.dt.month == month) & (template.time.dt.day == day)
        template[var].data[mask] = debiased_doy 
        logger.info(f'Debiased data for {date} added to dataset!')

        # Check that data was actaully replaced
        temp_masked = template[var].where(mask, drop = True)
        raw_masked = raw[var].where(mask, drop = True)
        bias = raw_masked - temp_masked # Back into data's bias
        if (bias.all() == 0): # Ensure it's not all zeros
            logger.error('Bias corrected data was not successfully applied to dataset.')
            return None
        else:
            logger.info(f'Bias for {date} successfully corrected.')

        # Close unnecessay datasets
        temp_masked.close()
        raw_masked.close()

    return template

def nan_test(var):
    """
    Returns location of Nans in debiased data.
    """
    
    fpath = glob.glob(str(current_dir / 'wrfout' / f'*{var}*.nc'))
    debiased = xr.open_dataset(fpath[0], decode_times = True)

    nan_data = debiased[var].where(debiased[var].isnull(), drop = True)

    return nan_data

# Catch silent errors and report to log file
@logger.catch 
def main(var, data_location, save = True):
    # Open datasets
    obs_path = glob.glob(str(data_location / 'gridMET' / f'*{var}*.nc'))
    obs = xr.open_dataset(obs_path[0], decode_times = True)

    raw_path = glob.glob(str(data_location / 'daily' / f'*{var}*.nc'))
    raw = xr.open_dataset(raw_path[0], decode_times = True)

    # Set location of interest as SLC airport
    lat = 40.788
    lon = -111.978

    # Pass datasets to debiaser
    test = ECDFM(obs, raw, var, lat, lon)

    if save:
        # Create output directory to store new cleaned files
        output_dir = current_dir / 'wrfout' 
        os.makedirs(output_dir, exist_ok = True) # Don't make if it already exists

        # Generate save name for debiased data
        out_path = os.path.join(output_dir, f'wrfout_GSLBIP_multimodel_ssp245_{var}_test.nc')

        # Save data to netCDF file
        test.to_netcdf(out_path)
        logger.success(f'File saved to: {out_path}')

    # Close out of data
    test.close()

# if __name__ == '__main__':

#     # Track program time in log files
#     start = time.perf_counter()
#     logger.info('Beginning execution.')

#     # Only inputs required
#     main(
#             var = 'tmmx',
#             data_location = current_dir,
#             save = False
#             )

#     # Report of runtime at completion 
#     logger.success(f'Debiasing process completed!')
#     logger.info(f'Total runtime: {time.perf_counter() - start:.4f}s')

#     # Force script to stop running once code is finished
#     sys.exit(0)

var = 'tmmx'
# Open datasets
obs_path = glob.glob(str(current_dir / 'gridMET' / f'*{var}*.nc'))
obs = xr.open_dataset(obs_path[0], decode_times = True)

raw_path = glob.glob(str(current_dir / 'daily' / f'*{var}*.nc'))
raw = xr.open_dataset(raw_path[0], decode_times = True)

# Set location of interest as SLC airport
lat = 40.788
lon = -111.978

# Slice raw data down to historical period
hist = raw.sel(time = slice('1985-01-01', '2014-12-31'))

# Select specific location of interest (choses nearest possible location)
obs_loc = loc_sel(obs, lat = lat, lon = lon)
logger.info(f'NaNs at selected obs location: {float(obs_loc[var].isnull().sum().values)}')
hist_loc = loc_sel(hist, lat = lat, lon = lon)
logger.info(f'NaNs at selected hist location: {float(hist_loc[var].isnull().sum().values)}')
raw_loc = loc_sel(raw, lat = lat, lon = lon)
logger.info(f'NaNs at selected raw location: {float(raw_loc[var].isnull().sum().values)}')
logger.info(raw_loc[var])

# Make copy of raw_loc to plug debiased data into
template = raw_loc.copy()

# Create date range that spans a calendar year
date = pd.to_datetime('2013-11-02') 

# Slice data into 31 day running windows centered on given date (note: 31 days is defualt setting)
obs_sliced = running_window_slice(obs, date)
logger.info(f'NaNs in obs running window: {float(obs_sliced[var].isnull().sum().values)}')
hist_sliced = running_window_slice(hist, date)
logger.info(f'NaNs in hist running window: {float(hist_sliced[var].isnull().sum().values)}')
raw_sliced = running_window_slice(raw, date)
logger.info(f'NaNs in raw running window: {float(raw_sliced[var].isnull().sum().values)}')
logger.info('Datasets sliced into running windows.')

# Create CDFs from sliced datasets
obs_cdf = ECDF(obs_sliced[var].values.flatten())
hist_cdf = ECDF(hist_sliced[var].values.flatten())
raw_cdf = ECDF(raw_sliced[var].values.flatten())

# Select day of interest and create mask
day = int(date.strftime('%d'))
month = int(date.strftime('%m'))
mask = (raw.time.dt.month == month) & (raw.time.dt.day == day)

# Apply mask to dataset
raw_date = raw.where(mask, drop = True) # This should select day of interest across all years in ds
logger.info(f'NaNs in raw data for {date}: {float(raw_date[var].isnull().sum().values)}')
logger.info(f'Dataset sliced down to date of interest: {date}')
logger.info(raw_date)

# Use raw_cdf like a function to find all of the percentiles in raw_date
raw_percentiles = raw_cdf(raw_date[var].values)
logger.info(f'Percentile values for raw data: {raw_percentiles}')
logger.info(f'NaNs in raw_percentiles: {float(np.isnan(raw_percentiles).sum())}')

# TODO: NaNs are appearing in obs and hist from this point on
# Invert hist_cdf and obs_cdf to take a percentile and output a data point
inv_obs = invert_ecdf(obs_cdf)
inv_hist = invert_ecdf(hist_cdf)

# Find data points for every percentile in raw_percentiles
obs_vals = inv_obs(raw_percentiles)
logger.info(f'NaNs in obs_vals: {float(np.isnan(obs_vals).sum())}')
logger.info(f'Obs values for given percentiles: {obs_vals}')
hist_vals = inv_hist(raw_percentiles)

15:20:32 |     INFO | plots.validation:loc_sel:line 89 - Selecting nearest location to 40.788, -111.978.
/uufs/chpc.utah.edu/common/home/strong-group7/sydney/miniforge3_envs/olympics/lib/python3.11/site-packages/xarray/core/dataarray.py:6318: FutureWarning: Behaviour of argmin/argmax with neither dim nor axis argument will change to return a dict of indices of each dimension. To get a single, flat index, please use np.argmin(da.data) or np.argmax(da.data) instead of da.argmin() or da.argmax().
  result = self.variable.argmin(dim, axis, keep_attrs, skipna)
15:20:32 |     INFO | plots.validation:loc_sel:line 100 - Point selected is <xarray.DataArray ()> Size: 8B
array(3916)
Coordinates:
    time     datetime64[ns] 8B 1985-01-01 from 40.788, -111.978.
15:20:32 |     INFO | plots.validation:loc_sel:line 102 - Min distance is located at 46, 43.
15:20:32 |     INFO | plots.validation:loc_sel:line 113 - New lat: 40.78413391113281
15:20:32 |     INFO | plots.validation:loc_sel:line 114 - New l

In [ ]:
np.isnan(obs_vals).sum()

np.int64(0)

In [ ]:
"""
Author: Sydney Smith
Date Created: September 29, 2026
"""

from datetime import datetime
import gc
import glob
from loguru import logger
import numpy as np
import os
import pandas as pd
from pathlib import Path
from scipy.interpolate import interp1d
from statsmodels.distributions.empirical_distribution import ECDF, monotone_fn_inverter
import sys
import time
import xarray as xr

# ==================================
# - Establish Relative File Path - 
# ==================================

current_dir = Path(__file__).resolve().parent
sys.path.append(str(current_dir))

from plots.validation import running_window_slice, loc_sel

# ===================
# - Set Up Logger - 
# ==================

# Create directory for log files if it doesn't already exist
log_path = str(current_dir / 'log')
os.makedirs(log_path, exist_ok = True)

# String filename
log_filename = datetime.now().strftime('log/%Y-%m-%d_%H-%M-%S.log')

# Custom format for log prints
log_format = log_format = '<cyan>{time:HH:mm:ss}</cyan> | <level>{level:>8}</level> | <yellow>{name}</yellow>:<cyan>{function}</cyan>:line <magenta>{line}</magenta> - <level>{message}</level>'

# Removes the default stderr sink
logger.remove()  
# Anything above info to console
logger.add(sys.stderr, colorize = True, format = log_format, level = 'INFO') 
# Anything above debug to log file
logger.add(log_filename, colorize = False, format = log_format, level = 'DEBUG')
logger.debug(f'Log files saved to {log_path}')

# =====================
# ---- Functions ----
# =====================

def invert_ecdf(ecdf_obj):
    """
    Inverts a statsmodels ECDF object using its .y and .x attributes, regardless of if
    the values are monotonic or not, with safe extrapolation to prevent NaNs.
    """
    # ECDF from statsmodels sets first x value to -inf which causes NaNs for very small percentiles passed to the inverted function
    # Mask out any indefinite values to prevent possible NaNs
    finite_mask = np.isfinite(ecdf_obj.x)
    x_vals = ecdf_obj.x[finite_mask]
    y_vals = ecdf_obj.y[finite_mask]

    # Ensure x_vals are unique and strictly increasing for interp1d (monotonic)
    x_unique, indices = np.unique(x_vals, return_index=True)
    y_unique = y_vals[indices]

    return interp1d(
        y_unique, # Pass y values as x values
        x_unique, # And vice versa
        bounds_error = False, 
        fill_value = 'extrapolate'  # Prevents NaNs if raw percentiles push past bounds
    )

def apply_to_date(obs: xr.Dataset, hist, raw, var, date) -> np.ndarray:
    """
    Debiases data for a given date by constructing a CDF using a dataset restricted to be within a 
    running window. Date must be a valid date that exists within the dataset formatted as '%Y-%m-%d' 
    datetime object.
    """
    # TODO: check how running_window_slice handles leap years (try to set calendar year to 1988 instead since it's a leap year)

    # Slice data into 31 day running windows centered on given date (note: 31 days is defualt setting)
    obs_sliced = running_window_slice(obs, date)
    logger.info(f'NaNs in obs running window: {float(obs_sliced[var].isnull().sum().values)}')

    hist_sliced = running_window_slice(hist, date)
    logger.info(f'NaNs in hist running window: {float(hist_sliced[var].isnull().sum().values)}')

    raw_sliced = running_window_slice(raw, date)
    logger.info(f'NaNs in raw running window: {float(raw_sliced[var].isnull().sum().values)}')
    logger.info('Datasets sliced into running windows.')

    # End function if NaNs are present in the source data
    if obs_sliced[var].isnull().any() or hist_sliced[var].isnull().any() or raw_sliced[var].isnull().any():
        logger.error('NaNs detected in source data.')
        return NotImplementedError

    # Create CDFs from sliced datasets
    obs_cdf = ECDF(obs_sliced[var].values.flatten())
    hist_cdf = ECDF(hist_sliced[var].values.flatten())
    raw_cdf = ECDF(raw_sliced[var].values.flatten())

    # Select day of interest and create mask
    day = int(date.strftime('%d'))
    month = int(date.strftime('%m'))
    mask = (raw.time.dt.month == month) & (raw.time.dt.day == day)

    # Skip dates that don't exist
    if not mask.any():
        logger.warning(f'No date found for {date}.')
        return None
    
    # Apply mask to dataset
    raw_date = raw.where(mask, drop = True) # This should select day of interest across all years in ds
    logger.info(f'Dataset sliced down to date of interest: {date}')
    logger.info(raw_date)

    # Use raw_cdf like a function to find all of the percentiles in raw_date
    raw_percentiles = raw_cdf(raw_date[var].values)
    logger.info(f'Percentile values for raw data: {raw_percentiles}')
    logger.info(f'NaNs in raw_percentiles: {float(np.isnan(raw_percentiles).sum())}')

    # TODO: NaNs are appearing in obs and hist from this point on
    # Invert hist_cdf and obs_cdf to take a percentile and output a data point
    inv_obs = invert_ecdf(obs_cdf)
    inv_hist = invert_ecdf(hist_cdf)

    # Find data points for every percentile in raw_percentiles
    obs_vals = inv_obs(raw_percentiles)
    logger.info(f'NaNs in obs_vals: {float(np.isnan(obs_vals).sum())}')
    logger.info(f'Obs values for given percentiles: {obs_vals}')

    hist_vals = inv_hist(raw_percentiles)
    logger.info(f'NaNs in hist_vals: {float(np.isnan(hist_vals).sum())}')
    logger.info(f'Historical values for given percentiles: {hist_vals}')

    # Perform bias adjustment
    bias = hist_vals - obs_vals
    logger.info(f'NaNs in bias: {float(np.isnan(bias).sum())}')
    raw_bias_corrected = raw_date[var].values - bias

    # End script early if there are Nans in data
    if np.isnan(raw_bias_corrected).any():
        logger.error(f'NaNs in debiased data: {float(np.isnan(raw_bias_corrected).sum())}')
        logger.info(f'Nan error occured on {date}.')
        return None

    logger.info('Raw data successfully bias corrected!')
    logger.info(raw_bias_corrected)
    logger.info(f'Bias corrected max: {raw_bias_corrected.max()}')
    logger.info(f'Bias corrected min: {raw_bias_corrected.min()}')

    # Close out of unwanted datasets
    obs_sliced.close()
    hist_sliced.close()
    raw_sliced.close()
    raw_date.close()
    del obs_vals
    del hist_vals
    del raw_percentiles
    gc.collect()

    return raw_bias_corrected

def ECDFM(obs, raw, var, lat, lon):
    """
    Performs ECDFM quantile mapping debiasing process on given data based on data from a given histroical
    period and corresponding observational data. Raw data should include both historical and future data. 
    """

    # Slice raw data down to historical period
    hist = raw.sel(time = slice('1985-01-01', '2014-12-31'))
    
    # Select specific location of interest (choses nearest possible location)
    obs_loc = loc_sel(obs, lat = lat, lon = lon)
    logger.info(f'NaNs at selected obs location: {float(obs_loc[var].isnull().sum().values)}')
    hist_loc = loc_sel(hist, lat = lat, lon = lon)
    logger.info(f'NaNs at selected hist location: {float(hist_loc[var].isnull().sum().values)}')
    raw_loc = loc_sel(raw, lat = lat, lon = lon)
    logger.info(f'NaNs at selected raw location: {float(raw_loc[var].isnull().sum().values)}')
    logger.info(raw_loc[var])

    # Make copy of raw_loc to plug debiased data into
    template = raw_loc.copy()

    # Create date range that spans a calendar year
    dates = pd.date_range(start = '1985-01-01', end = '1985-12-31', freq = 'D') 

    for date in dates:
        # Apply bias correction for single date
        logger.info(f'Date type: {type(date)}')
        debiased_doy = apply_to_date(obs_loc, hist_loc, raw_loc, var, date)

        # Exit out of function if Nans occur
        if debiased_doy is None:
            logger.error(f'Nans detected in dataset. See log above for more information.')
            return None

        # Pull month and day from date
        day = int(date.strftime('%d'))
        month = int(date.strftime('%m'))

        # Plug in debiased data
        mask = (template.time.dt.month == month) & (template.time.dt.day == day)
        template[var].data[mask] = debiased_doy 
        logger.info(f'Debiased data for {date} added to dataset!')

        # Check that data was actaully replaced
        temp_masked = template[var].where(mask, drop = True)
        raw_masked = raw[var].where(mask, drop = True)
        bias = raw_masked - temp_masked # Back into data's bias
        if (bias.all() == 0): # Ensure it's not all zeros
            logger.error('Bias corrected data was not successfully applied to dataset.')
            return None
        else:
            logger.info(f'Bias for {date} successfully corrected.')

        # Close unnecessay datasets
        temp_masked.close()
        raw_masked.close()

    return template

def nan_test(var):
    """
    Returns location of Nans in debiased data.
    """
    
    fpath = glob.glob(str(current_dir / 'wrfout' / f'*{var}*.nc'))
    debiased = xr.open_dataset(fpath[0], decode_times = True)

    nan_data = debiased[var].where(debiased[var].isnull(), drop = True)

    return nan_data

def np_nan_loc(arr):
    """
    Find locations of NaNs in np.array.
    """
    return np.argwhere(np.isnan(arr))

# Catch silent errors and report to log file
@logger.catch 
def main(var, data_location, save = True):
    # Open datasets
    obs_path = glob.glob(str(data_location / 'gridMET' / f'*{var}*.nc'))
    obs = xr.open_dataset(obs_path[0], decode_times = True)

    raw_path = glob.glob(str(data_location / 'daily' / f'*{var}*.nc'))
    raw = xr.open_dataset(raw_path[0], decode_times = True)

    # Set location of interest as SLC airport
    lat = 40.788
    lon = -111.978

    # Pass datasets to debiaser
    test = ECDFM(obs, raw, var, lat, lon)

    if save:
        # Create output directory to store new cleaned files
        output_dir = current_dir / 'wrfout' 
        os.makedirs(output_dir, exist_ok = True) # Don't make if it already exists

        # Generate save name for debiased data
        out_path = os.path.join(output_dir, f'wrfout_GSLBIP_multimodel_ssp245_{var}_test.nc')

        # Save data to netCDF file
        test.to_netcdf(out_path)
        logger.success(f'File saved to: {out_path}')

    # Close out of data
    test.close()

# if __name__ == '__main__':

#     # Track program time in log files
#     start = time.perf_counter()
#     logger.info('Beginning execution.')

#     # Only inputs required
#     main(
#             var = 'tmmx',
#             data_location = current_dir,
#             save = True
#             )

#     # Report of runtime at completion 
#     logger.success(f'Debiasing process completed!')
#     logger.info(f'Total runtime: {time.perf_counter() - start:.4f}s')

#     # Force script to stop running once code is finished
#     sys.exit(0)


var = 'tmmx'
# Open datasets
obs_path = glob.glob(str(current_dir / 'gridMET' / f'*{var}*.nc'))
obs = xr.open_dataset(obs_path[0], decode_times = True)

raw_path = glob.glob(str(current_dir / 'daily' / f'*{var}*.nc'))
raw = xr.open_dataset(raw_path[0], decode_times = True)

# Set location of interest as SLC airport
lat = 40.788
lon = -111.978


# Slice raw data down to historical period
hist = raw.sel(time = slice('1985-01-01', '2014-12-31'))

# Select specific location of interest (choses nearest possible location)
obs_loc = loc_sel(obs, lat = lat, lon = lon)
logger.info(f'NaNs at selected obs location: {float(obs_loc[var].isnull().sum().values)}')
hist_loc = loc_sel(hist, lat = lat, lon = lon)
logger.info(f'NaNs at selected hist location: {float(hist_loc[var].isnull().sum().values)}')
raw_loc = loc_sel(raw, lat = lat, lon = lon)
logger.info(f'NaNs at selected raw location: {float(raw_loc[var].isnull().sum().values)}')
logger.info(raw_loc[var])

# Make copy of raw_loc to plug debiased data into
template = raw_loc.copy()

# Create date range that spans a calendar year
dates = pd.to_datetime(start = '1985-04-16') 

for date in dates:
    # Apply bias correction for single date
    logger.info(f'Date type: {type(date)}')
    debiased_doy = apply_to_date(obs_loc, hist_loc, raw_loc, var, date)

    # Exit out of function if Nans occur
    if debiased_doy is None:
        logger.error(f'Nans detected in dataset. See log above for more information.')

    # Pull month and day from date
    day = int(date.strftime('%d'))
    month = int(date.strftime('%m'))

    # Plug in debiased data
    mask = (template.time.dt.month == month) & (template.time.dt.day == day)
    template[var].data[mask] = debiased_doy 
    logger.info(f'Debiased data for {date} added to dataset!')

    # Check that data was actaully replaced
    temp_masked = template[var].where(mask, drop = True)
    raw_masked = raw[var].where(mask, drop = True)
    bias = raw_masked - temp_masked # Back into data's bias
    if (bias.all() == 0): # Ensure it's not all zeros
        logger.error('Bias corrected data was not successfully applied to dataset.')
    else:
        logger.info(f'Bias for {date} successfully corrected.')

16:00:06 |     INFO | plots.validation:loc_sel:line 89 - Selecting nearest location to 40.788, -111.978.
/uufs/chpc.utah.edu/common/home/strong-group7/sydney/miniforge3_envs/olympics/lib/python3.11/site-packages/xarray/core/dataarray.py:6318: FutureWarning: Behaviour of argmin/argmax with neither dim nor axis argument will change to return a dict of indices of each dimension. To get a single, flat index, please use np.argmin(da.data) or np.argmax(da.data) instead of da.argmin() or da.argmax().
  result = self.variable.argmin(dim, axis, keep_attrs, skipna)
16:00:06 |     INFO | plots.validation:loc_sel:line 100 - Point selected is <xarray.DataArray ()> Size: 8B
array(3916)
Coordinates:
    time     datetime64[ns] 8B 1985-01-01 from 40.788, -111.978.
16:00:06 |     INFO | plots.validation:loc_sel:line 102 - Min distance is located at 46, 43.
16:00:06 |     INFO | plots.validation:loc_sel:line 113 - New lat: 40.78413391113281
16:00:06 |     INFO | plots.validation:loc_sel:line 114 - New l

TypeError: to_datetime() got an unexpected keyword argument 'start'

In [ ]:
"""
Author: Sydney Smith
Date Created: September 29, 2026
"""

from datetime import datetime
import gc
import glob
from loguru import logger
import numpy as np
import os
import pandas as pd
from pathlib import Path
from scipy.interpolate import interp1d
from statsmodels.distributions.empirical_distribution import ECDF, monotone_fn_inverter
import sys
import time
import xarray as xr

# ==================================
# - Establish Relative File Path - 
# ==================================

current_dir = Path(__file__).resolve().parent
sys.path.append(str(current_dir))

from plots.validation import running_window_slice, loc_sel

# ===================
# - Set Up Logger - 
# ==================

# Create directory for log files if it doesn't already exist
log_path = str(current_dir / 'log')
os.makedirs(log_path, exist_ok = True)

# String filename
log_filename = datetime.now().strftime('log/%Y-%m-%d_%H-%M-%S.log')

# Custom format for log prints
log_format = log_format = '<cyan>{time:HH:mm:ss}</cyan> | <level>{level:>8}</level> | <yellow>{name}</yellow>:<cyan>{function}</cyan>:line <magenta>{line}</magenta> - <level>{message}</level>'

# Removes the default stderr sink
logger.remove()  
# Anything above info to console
logger.add(sys.stderr, colorize = True, format = log_format, level = 'INFO') 
# Anything above debug to log file
logger.add(log_filename, colorize = False, format = log_format, level = 'DEBUG')
logger.debug(f'Log files saved to {log_path}')

# =====================
# ---- Functions ----
# =====================

def invert_ecdf(ecdf_obj):
    """
    Inverts a statsmodels ECDF object using its .y and .x attributes, regardless of if
    the values are monotonic or not, with safe extrapolation to prevent NaNs.
    """
    # ECDF from statsmodels sets first x value to -inf which causes NaNs for very small percentiles passed to the inverted function
    # Mask out any indefinite values to prevent possible NaNs
    finite_mask = np.isfinite(ecdf_obj.x)
    x_vals = ecdf_obj.x[finite_mask]
    y_vals = ecdf_obj.y[finite_mask]

    # Ensure x_vals are unique and strictly increasing for interp1d (monotonic)
    x_unique, indices = np.unique(x_vals, return_index=True)
    y_unique = y_vals[indices]

    return interp1d(
        y_unique, # Pass y values as x values
        x_unique, # And vice versa
        bounds_error = False, 
        fill_value = 'extrapolate'  # Prevents NaNs if raw percentiles push past bounds
    )

def apply_to_date(obs: xr.Dataset, hist, raw, var, date) -> np.ndarray:
    """
    Debiases data for a given date by constructing a CDF using a dataset restricted to be within a 
    running window. Date must be a valid date that exists within the dataset formatted as '%Y-%m-%d' 
    datetime object.
    """
    # TODO: check how running_window_slice handles leap years (try to set calendar year to 1988 instead since it's a leap year)

    # Slice data into 31 day running windows centered on given date (note: 31 days is defualt setting)
    obs_sliced = running_window_slice(obs, date)
    logger.info(f'NaNs in obs running window: {float(obs_sliced[var].isnull().sum().values)}')

    hist_sliced = running_window_slice(hist, date)
    logger.info(f'NaNs in hist running window: {float(hist_sliced[var].isnull().sum().values)}')

    raw_sliced = running_window_slice(raw, date)
    logger.info(f'NaNs in raw running window: {float(raw_sliced[var].isnull().sum().values)}')
    logger.info('Datasets sliced into running windows.')

    # End function if NaNs are present in the source data
    if obs_sliced[var].isnull().any() or hist_sliced[var].isnull().any() or raw_sliced[var].isnull().any():
        logger.error('NaNs detected in source data.')
        return NotImplementedError

    # Create CDFs from sliced datasets
    obs_cdf = ECDF(obs_sliced[var].values.flatten())
    hist_cdf = ECDF(hist_sliced[var].values.flatten())
    raw_cdf = ECDF(raw_sliced[var].values.flatten())

    # Select day of interest and create mask
    day = int(date.strftime('%d'))
    month = int(date.strftime('%m'))
    mask = (raw.time.dt.month == month) & (raw.time.dt.day == day)

    # Skip dates that don't exist
    if not mask.any():
        logger.warning(f'No date found for {date}.')
        return None
    
    # Apply mask to dataset
    raw_date = raw.where(mask, drop = True) # This should select day of interest across all years in ds
    logger.info(f'Dataset sliced down to date of interest: {date}')
    logger.info(raw_date)

    # Use raw_cdf like a function to find all of the percentiles in raw_date
    raw_percentiles = raw_cdf(raw_date[var].values)
    logger.info(f'Percentile values for raw data: {raw_percentiles}')
    logger.info(f'NaNs in raw_percentiles: {float(np.isnan(raw_percentiles).sum())}')

    # TODO: NaNs are appearing in obs and hist from this point on
    # Invert hist_cdf and obs_cdf to take a percentile and output a data point
    inv_obs = invert_ecdf(obs_cdf)
    inv_hist = invert_ecdf(hist_cdf)

    # Find data points for every percentile in raw_percentiles
    obs_vals = inv_obs(raw_percentiles)
    logger.info(f'NaNs in obs_vals: {float(np.isnan(obs_vals).sum())}')
    logger.info(f'Obs values for given percentiles: {obs_vals}')

    hist_vals = inv_hist(raw_percentiles)
    logger.info(f'NaNs in hist_vals: {float(np.isnan(hist_vals).sum())}')
    logger.info(f'Historical values for given percentiles: {hist_vals}')

    # Perform bias adjustment
    bias = hist_vals - obs_vals
    logger.info(f'NaNs in bias: {float(np.isnan(bias).sum())}')
    raw_bias_corrected = raw_date[var].values - bias

    # End script early if there are Nans in data
    if np.isnan(raw_bias_corrected).any():
        logger.error(f'NaNs in debiased data: {float(np.isnan(raw_bias_corrected).sum())}')
        logger.info(f'Nan error occured on {date}.')
        return None

    logger.info('Raw data successfully bias corrected!')
    logger.info(raw_bias_corrected)
    logger.info(f'Bias corrected max: {raw_bias_corrected.max()}')
    logger.info(f'Bias corrected min: {raw_bias_corrected.min()}')

    # Close out of unwanted datasets
    obs_sliced.close()
    hist_sliced.close()
    raw_sliced.close()
    raw_date.close()
    del obs_vals
    del hist_vals
    del raw_percentiles
    gc.collect()

    return raw_bias_corrected

def ECDFM(obs, raw, var, lat, lon):
    """
    Performs ECDFM quantile mapping debiasing process on given data based on data from a given histroical
    period and corresponding observational data. Raw data should include both historical and future data. 
    """

    # Slice raw data down to historical period
    hist = raw.sel(time = slice('1985-01-01', '2014-12-31'))
    
    # Select specific location of interest (choses nearest possible location)
    obs_loc = loc_sel(obs, lat = lat, lon = lon)
    logger.info(f'NaNs at selected obs location: {float(obs_loc[var].isnull().sum().values)}')
    hist_loc = loc_sel(hist, lat = lat, lon = lon)
    logger.info(f'NaNs at selected hist location: {float(hist_loc[var].isnull().sum().values)}')
    raw_loc = loc_sel(raw, lat = lat, lon = lon)
    logger.info(f'NaNs at selected raw location: {float(raw_loc[var].isnull().sum().values)}')
    logger.info(raw_loc[var])

    # Make copy of raw_loc to plug debiased data into
    template = raw_loc.copy()

    # Create date range that spans a calendar year
    dates = pd.date_range(start = '1985-01-01', end = '1985-12-31', freq = 'D') 

    for date in dates:
        # Apply bias correction for single date
        logger.info(f'Date type: {type(date)}')
        debiased_doy = apply_to_date(obs_loc, hist_loc, raw_loc, var, date)

        # Exit out of function if Nans occur
        if debiased_doy is None:
            logger.error(f'Nans detected in dataset. See log above for more information.')
            return None

        # Pull month and day from date
        day = int(date.strftime('%d'))
        month = int(date.strftime('%m'))

        # Plug in debiased data
        mask = (template.time.dt.month == month) & (template.time.dt.day == day)
        template[var].data[mask] = debiased_doy 
        logger.info(f'Debiased data for {date} added to dataset!')

        # Check that data was actaully replaced
        temp_masked = template[var].where(mask, drop = True)
        raw_masked = raw[var].where(mask, drop = True)
        bias = raw_masked - temp_masked # Back into data's bias
        if (bias.all() == 0): # Ensure it's not all zeros
            logger.error('Bias corrected data was not successfully applied to dataset.')
            return None
        else:
            logger.info(f'Bias for {date} successfully corrected.')

        # Close unnecessay datasets
        temp_masked.close()
        raw_masked.close()

    return template

def nan_test(var):
    """
    Returns location of Nans in debiased data.
    """
    
    fpath = glob.glob(str(current_dir / 'wrfout' / f'*{var}*.nc'))
    debiased = xr.open_dataset(fpath[0], decode_times = True)

    nan_data = debiased[var].where(debiased[var].isnull(), drop = True)

    return nan_data

def np_nan_loc(arr):
    """
    Find locations of NaNs in np.array.
    """
    return np.argwhere(np.isnan(arr))

# Catch silent errors and report to log file
@logger.catch 
def main(var, data_location, save = True):
    # Open datasets
    obs_path = glob.glob(str(data_location / 'gridMET' / f'*{var}*.nc'))
    obs = xr.open_dataset(obs_path[0], decode_times = True)

    raw_path = glob.glob(str(data_location / 'daily' / f'*{var}*.nc'))
    raw = xr.open_dataset(raw_path[0], decode_times = True)

    # Set location of interest as SLC airport
    lat = 40.788
    lon = -111.978

    # Pass datasets to debiaser
    test = ECDFM(obs, raw, var, lat, lon)

    if save:
        # Create output directory to store new cleaned files
        output_dir = current_dir / 'wrfout' 
        os.makedirs(output_dir, exist_ok = True) # Don't make if it already exists

        # Generate save name for debiased data
        out_path = os.path.join(output_dir, f'wrfout_GSLBIP_multimodel_ssp245_{var}_test.nc')

        # Save data to netCDF file
        test.to_netcdf(out_path)
        logger.success(f'File saved to: {out_path}')

    # Close out of data
    test.close()

# if __name__ == '__main__':

#     # Track program time in log files
#     start = time.perf_counter()
#     logger.info('Beginning execution.')

#     # Only inputs required
#     main(
#             var = 'tmmx',
#             data_location = current_dir,
#             save = True
#             )

#     # Report of runtime at completion 
#     logger.success(f'Debiasing process completed!')
#     logger.info(f'Total runtime: {time.perf_counter() - start:.4f}s')

#     # Force script to stop running once code is finished
#     sys.exit(0)


var = 'tmmx'
# Open datasets
obs_path = glob.glob(str(current_dir / 'gridMET' / f'*{var}*.nc'))
obs = xr.open_dataset(obs_path[0], decode_times = True)

raw_path = glob.glob(str(current_dir / 'daily' / f'*{var}*.nc'))
raw = xr.open_dataset(raw_path[0], decode_times = True)

# Set location of interest as SLC airport
lat = 40.788
lon = -111.978


# Slice raw data down to historical period
hist = raw.sel(time = slice('1985-01-01', '2014-12-31'))

# Select specific location of interest (choses nearest possible location)
obs_loc = loc_sel(obs, lat = lat, lon = lon)
logger.info(f'NaNs at selected obs location: {float(obs_loc[var].isnull().sum().values)}')
hist_loc = loc_sel(hist, lat = lat, lon = lon)
logger.info(f'NaNs at selected hist location: {float(hist_loc[var].isnull().sum().values)}')
raw_loc = loc_sel(raw, lat = lat, lon = lon)
logger.info(f'NaNs at selected raw location: {float(raw_loc[var].isnull().sum().values)}')
logger.info(raw_loc[var])

# Make copy of raw_loc to plug debiased data into
template = raw_loc.copy()

# Create date range that spans a calendar year
dates = pd.to_datetime('1985-04-16') 

for date in dates:
    # Apply bias correction for single date
    logger.info(f'Date type: {type(date)}')
    debiased_doy = apply_to_date(obs_loc, hist_loc, raw_loc, var, date)

    # Exit out of function if Nans occur
    if debiased_doy is None:
        logger.error(f'Nans detected in dataset. See log above for more information.')

    # Pull month and day from date
    day = int(date.strftime('%d'))
    month = int(date.strftime('%m'))

    # Plug in debiased data
    mask = (template.time.dt.month == month) & (template.time.dt.day == day)
    template[var].data[mask] = debiased_doy 
    logger.info(f'Debiased data for {date} added to dataset!')

    # Check that data was actaully replaced
    temp_masked = template[var].where(mask, drop = True)
    raw_masked = raw[var].where(mask, drop = True)
    bias = raw_masked - temp_masked # Back into data's bias
    if (bias.all() == 0): # Ensure it's not all zeros
        logger.error('Bias corrected data was not successfully applied to dataset.')
    else:
        logger.info(f'Bias for {date} successfully corrected.')

16:00:19 |     INFO | plots.validation:loc_sel:line 89 - Selecting nearest location to 40.788, -111.978.
/uufs/chpc.utah.edu/common/home/strong-group7/sydney/miniforge3_envs/olympics/lib/python3.11/site-packages/xarray/core/dataarray.py:6318: FutureWarning: Behaviour of argmin/argmax with neither dim nor axis argument will change to return a dict of indices of each dimension. To get a single, flat index, please use np.argmin(da.data) or np.argmax(da.data) instead of da.argmin() or da.argmax().
  result = self.variable.argmin(dim, axis, keep_attrs, skipna)
16:00:19 |     INFO | plots.validation:loc_sel:line 100 - Point selected is <xarray.DataArray ()> Size: 8B
array(3916)
Coordinates:
    time     datetime64[ns] 8B 1985-01-01 from 40.788, -111.978.
16:00:19 |     INFO | plots.validation:loc_sel:line 102 - Min distance is located at 46, 43.
16:00:19 |     INFO | plots.validation:loc_sel:line 113 - New lat: 40.78413391113281
16:00:19 |     INFO | plots.validation:loc_sel:line 114 - New l

TypeError: 'Timestamp' object is not iterable

In [ ]:
"""
Author: Sydney Smith
Date Created: September 29, 2026
"""

from datetime import datetime
import gc
import glob
from loguru import logger
import numpy as np
import os
import pandas as pd
from pathlib import Path
from scipy.interpolate import interp1d
from statsmodels.distributions.empirical_distribution import ECDF, monotone_fn_inverter
import sys
import time
import xarray as xr

# ==================================
# - Establish Relative File Path - 
# ==================================

current_dir = Path(__file__).resolve().parent
sys.path.append(str(current_dir))

from plots.validation import running_window_slice, loc_sel

# ===================
# - Set Up Logger - 
# ==================

# Create directory for log files if it doesn't already exist
log_path = str(current_dir / 'log')
os.makedirs(log_path, exist_ok = True)

# String filename
log_filename = datetime.now().strftime('log/%Y-%m-%d_%H-%M-%S.log')

# Custom format for log prints
log_format = log_format = '<cyan>{time:HH:mm:ss}</cyan> | <level>{level:>8}</level> | <yellow>{name}</yellow>:<cyan>{function}</cyan>:line <magenta>{line}</magenta> - <level>{message}</level>'

# Removes the default stderr sink
logger.remove()  
# Anything above info to console
logger.add(sys.stderr, colorize = True, format = log_format, level = 'INFO') 
# Anything above debug to log file
logger.add(log_filename, colorize = False, format = log_format, level = 'DEBUG')
logger.debug(f'Log files saved to {log_path}')

# =====================
# ---- Functions ----
# =====================

def invert_ecdf(ecdf_obj):
    """
    Inverts a statsmodels ECDF object using its .y and .x attributes, regardless of if
    the values are monotonic or not, with safe extrapolation to prevent NaNs.
    """
    # ECDF from statsmodels sets first x value to -inf which causes NaNs for very small percentiles passed to the inverted function
    # Mask out any indefinite values to prevent possible NaNs
    finite_mask = np.isfinite(ecdf_obj.x)
    x_vals = ecdf_obj.x[finite_mask]
    y_vals = ecdf_obj.y[finite_mask]

    # Ensure x_vals are unique and strictly increasing for interp1d (monotonic)
    x_unique, indices = np.unique(x_vals, return_index=True)
    y_unique = y_vals[indices]

    return interp1d(
        y_unique, # Pass y values as x values
        x_unique, # And vice versa
        bounds_error = False, 
        fill_value = 'extrapolate'  # Prevents NaNs if raw percentiles push past bounds
    )

def apply_to_date(obs: xr.Dataset, hist, raw, var, date) -> np.ndarray:
    """
    Debiases data for a given date by constructing a CDF using a dataset restricted to be within a 
    running window. Date must be a valid date that exists within the dataset formatted as '%Y-%m-%d' 
    datetime object.
    """
    # TODO: check how running_window_slice handles leap years (try to set calendar year to 1988 instead since it's a leap year)

    # Slice data into 31 day running windows centered on given date (note: 31 days is defualt setting)
    obs_sliced = running_window_slice(obs, date)
    logger.info(f'NaNs in obs running window: {float(obs_sliced[var].isnull().sum().values)}')

    hist_sliced = running_window_slice(hist, date)
    logger.info(f'NaNs in hist running window: {float(hist_sliced[var].isnull().sum().values)}')

    raw_sliced = running_window_slice(raw, date)
    logger.info(f'NaNs in raw running window: {float(raw_sliced[var].isnull().sum().values)}')
    logger.info('Datasets sliced into running windows.')

    # End function if NaNs are present in the source data
    if obs_sliced[var].isnull().any() or hist_sliced[var].isnull().any() or raw_sliced[var].isnull().any():
        logger.error('NaNs detected in source data.')
        return NotImplementedError

    # Create CDFs from sliced datasets
    obs_cdf = ECDF(obs_sliced[var].values.flatten())
    hist_cdf = ECDF(hist_sliced[var].values.flatten())
    raw_cdf = ECDF(raw_sliced[var].values.flatten())

    # Select day of interest and create mask
    day = int(date.strftime('%d'))
    month = int(date.strftime('%m'))
    mask = (raw.time.dt.month == month) & (raw.time.dt.day == day)

    # Skip dates that don't exist
    if not mask.any():
        logger.warning(f'No date found for {date}.')
        return None
    
    # Apply mask to dataset
    raw_date = raw.where(mask, drop = True) # This should select day of interest across all years in ds
    logger.info(f'Dataset sliced down to date of interest: {date}')
    logger.info(raw_date)

    # Use raw_cdf like a function to find all of the percentiles in raw_date
    raw_percentiles = raw_cdf(raw_date[var].values)
    logger.info(f'Percentile values for raw data: {raw_percentiles}')
    logger.info(f'NaNs in raw_percentiles: {float(np.isnan(raw_percentiles).sum())}')

    # TODO: NaNs are appearing in obs and hist from this point on
    # Invert hist_cdf and obs_cdf to take a percentile and output a data point
    inv_obs = invert_ecdf(obs_cdf)
    inv_hist = invert_ecdf(hist_cdf)

    # Find data points for every percentile in raw_percentiles
    obs_vals = inv_obs(raw_percentiles)
    logger.info(f'NaNs in obs_vals: {float(np.isnan(obs_vals).sum())}')
    logger.info(f'Obs values for given percentiles: {obs_vals}')

    hist_vals = inv_hist(raw_percentiles)
    logger.info(f'NaNs in hist_vals: {float(np.isnan(hist_vals).sum())}')
    logger.info(f'Historical values for given percentiles: {hist_vals}')

    # Perform bias adjustment
    bias = hist_vals - obs_vals
    logger.info(f'NaNs in bias: {float(np.isnan(bias).sum())}')
    raw_bias_corrected = raw_date[var].values - bias

    # End script early if there are Nans in data
    if np.isnan(raw_bias_corrected).any():
        logger.error(f'NaNs in debiased data: {float(np.isnan(raw_bias_corrected).sum())}')
        logger.info(f'Nan error occured on {date}.')
        return None

    logger.info('Raw data successfully bias corrected!')
    logger.info(raw_bias_corrected)
    logger.info(f'Bias corrected max: {raw_bias_corrected.max()}')
    logger.info(f'Bias corrected min: {raw_bias_corrected.min()}')

    # Close out of unwanted datasets
    obs_sliced.close()
    hist_sliced.close()
    raw_sliced.close()
    raw_date.close()
    del obs_vals
    del hist_vals
    del raw_percentiles
    gc.collect()

    return raw_bias_corrected

def ECDFM(obs, raw, var, lat, lon):
    """
    Performs ECDFM quantile mapping debiasing process on given data based on data from a given histroical
    period and corresponding observational data. Raw data should include both historical and future data. 
    """

    # Slice raw data down to historical period
    hist = raw.sel(time = slice('1985-01-01', '2014-12-31'))
    
    # Select specific location of interest (choses nearest possible location)
    obs_loc = loc_sel(obs, lat = lat, lon = lon)
    logger.info(f'NaNs at selected obs location: {float(obs_loc[var].isnull().sum().values)}')
    hist_loc = loc_sel(hist, lat = lat, lon = lon)
    logger.info(f'NaNs at selected hist location: {float(hist_loc[var].isnull().sum().values)}')
    raw_loc = loc_sel(raw, lat = lat, lon = lon)
    logger.info(f'NaNs at selected raw location: {float(raw_loc[var].isnull().sum().values)}')
    logger.info(raw_loc[var])

    # Make copy of raw_loc to plug debiased data into
    template = raw_loc.copy()

    # Create date range that spans a calendar year
    dates = pd.date_range(start = '1985-01-01', end = '1985-12-31', freq = 'D') 

    for date in dates:
        # Apply bias correction for single date
        logger.info(f'Date type: {type(date)}')
        debiased_doy = apply_to_date(obs_loc, hist_loc, raw_loc, var, date)

        # Exit out of function if Nans occur
        if debiased_doy is None:
            logger.error(f'Nans detected in dataset. See log above for more information.')
            return None

        # Pull month and day from date
        day = int(date.strftime('%d'))
        month = int(date.strftime('%m'))

        # Plug in debiased data
        mask = (template.time.dt.month == month) & (template.time.dt.day == day)
        template[var].data[mask] = debiased_doy 
        logger.info(f'Debiased data for {date} added to dataset!')

        # Check that data was actaully replaced
        temp_masked = template[var].where(mask, drop = True)
        raw_masked = raw[var].where(mask, drop = True)
        bias = raw_masked - temp_masked # Back into data's bias
        if (bias.all() == 0): # Ensure it's not all zeros
            logger.error('Bias corrected data was not successfully applied to dataset.')
            return None
        else:
            logger.info(f'Bias for {date} successfully corrected.')

        # Close unnecessay datasets
        temp_masked.close()
        raw_masked.close()

    return template

def nan_test(var):
    """
    Returns location of Nans in debiased data.
    """
    
    fpath = glob.glob(str(current_dir / 'wrfout' / f'*{var}*.nc'))
    debiased = xr.open_dataset(fpath[0], decode_times = True)

    nan_data = debiased[var].where(debiased[var].isnull(), drop = True)

    return nan_data

def np_nan_loc(arr):
    """
    Find locations of NaNs in np.array.
    """
    return np.argwhere(np.isnan(arr))

# Catch silent errors and report to log file
@logger.catch 
def main(var, data_location, save = True):
    # Open datasets
    obs_path = glob.glob(str(data_location / 'gridMET' / f'*{var}*.nc'))
    obs = xr.open_dataset(obs_path[0], decode_times = True)

    raw_path = glob.glob(str(data_location / 'daily' / f'*{var}*.nc'))
    raw = xr.open_dataset(raw_path[0], decode_times = True)

    # Set location of interest as SLC airport
    lat = 40.788
    lon = -111.978

    # Pass datasets to debiaser
    test = ECDFM(obs, raw, var, lat, lon)

    if save:
        # Create output directory to store new cleaned files
        output_dir = current_dir / 'wrfout' 
        os.makedirs(output_dir, exist_ok = True) # Don't make if it already exists

        # Generate save name for debiased data
        out_path = os.path.join(output_dir, f'wrfout_GSLBIP_multimodel_ssp245_{var}_test.nc')

        # Save data to netCDF file
        test.to_netcdf(out_path)
        logger.success(f'File saved to: {out_path}')

    # Close out of data
    test.close()

# if __name__ == '__main__':

#     # Track program time in log files
#     start = time.perf_counter()
#     logger.info('Beginning execution.')

#     # Only inputs required
#     main(
#             var = 'tmmx',
#             data_location = current_dir,
#             save = True
#             )

#     # Report of runtime at completion 
#     logger.success(f'Debiasing process completed!')
#     logger.info(f'Total runtime: {time.perf_counter() - start:.4f}s')

#     # Force script to stop running once code is finished
#     sys.exit(0)


var = 'tmmx'
# Open datasets
obs_path = glob.glob(str(current_dir / 'gridMET' / f'*{var}*.nc'))
obs = xr.open_dataset(obs_path[0], decode_times = True)

raw_path = glob.glob(str(current_dir / 'daily' / f'*{var}*.nc'))
raw = xr.open_dataset(raw_path[0], decode_times = True)

# Set location of interest as SLC airport
lat = 40.788
lon = -111.978


# Slice raw data down to historical period
hist = raw.sel(time = slice('1985-01-01', '2014-12-31'))

# Select specific location of interest (choses nearest possible location)
obs_loc = loc_sel(obs, lat = lat, lon = lon)
logger.info(f'NaNs at selected obs location: {float(obs_loc[var].isnull().sum().values)}')
hist_loc = loc_sel(hist, lat = lat, lon = lon)
logger.info(f'NaNs at selected hist location: {float(hist_loc[var].isnull().sum().values)}')
raw_loc = loc_sel(raw, lat = lat, lon = lon)
logger.info(f'NaNs at selected raw location: {float(raw_loc[var].isnull().sum().values)}')
logger.info(raw_loc[var])

# Make copy of raw_loc to plug debiased data into
template = raw_loc.copy()

# Create date range that spans a calendar year
date = pd.to_datetime('1985-04-16') 

# Apply bias correction for single date
logger.info(f'Date type: {type(date)}')
debiased_doy = apply_to_date(obs_loc, hist_loc, raw_loc, var, date)

# Exit out of function if Nans occur
if debiased_doy is None:
    logger.error(f'Nans detected in dataset. See log above for more information.')

# Pull month and day from date
day = int(date.strftime('%d'))
month = int(date.strftime('%m'))

# Plug in debiased data
mask = (template.time.dt.month == month) & (template.time.dt.day == day)
template[var].data[mask] = debiased_doy 
logger.info(f'Debiased data for {date} added to dataset!')

# Check that data was actaully replaced
temp_masked = template[var].where(mask, drop = True)
raw_masked = raw[var].where(mask, drop = True)
bias = raw_masked - temp_masked # Back into data's bias
if (bias.all() == 0): # Ensure it's not all zeros
    logger.error('Bias corrected data was not successfully applied to dataset.')
else:
    logger.info(f'Bias for {date} successfully corrected.')

16:00:40 |     INFO | plots.validation:loc_sel:line 89 - Selecting nearest location to 40.788, -111.978.
/uufs/chpc.utah.edu/common/home/strong-group7/sydney/miniforge3_envs/olympics/lib/python3.11/site-packages/xarray/core/dataarray.py:6318: FutureWarning: Behaviour of argmin/argmax with neither dim nor axis argument will change to return a dict of indices of each dimension. To get a single, flat index, please use np.argmin(da.data) or np.argmax(da.data) instead of da.argmin() or da.argmax().
  result = self.variable.argmin(dim, axis, keep_attrs, skipna)
16:00:40 |     INFO | plots.validation:loc_sel:line 100 - Point selected is <xarray.DataArray ()> Size: 8B
array(3916)
Coordinates:
    time     datetime64[ns] 8B 1985-01-01 from 40.788, -111.978.
16:00:40 |     INFO | plots.validation:loc_sel:line 102 - Min distance is located at 46, 43.
16:00:40 |     INFO | plots.validation:loc_sel:line 113 - New lat: 40.78413391113281
16:00:40 |     INFO | plots.validation:loc_sel:line 114 - New l

In [ ]:
template

<xarray.Dataset> Size: 2GB
Dimensions:  (time: 41888, lat: 111, lon: 90)
Coordinates:
  * time     (time) datetime64[ns] 335kB 1985-01-01 1985-01-02 ... 2099-12-30
    lat      (time) float32 168kB ...
    lon      (time) float32 168kB ...
Data variables:
    tmmx     (time, lat, lon) float32 2GB 268.8 269.5 269.9 ... 272.6 272.2
Attributes: (12/155)
    TITLE:                            OUTPUT FROM WRF V4.6.0 MODEL
    START_DATE:                      1984-01-01_06:00:00
    SIMULATION_START_DATE:           1984-01-01_06:00:00
    WEST-EAST_GRID_DIMENSION:        91
    SOUTH-NORTH_GRID_DIMENSION:      112
    BOTTOM-TOP_GRID_DIMENSION:       39
    ...                              ...
    ISLAKE:                          21
    ISICE:                           15
    ISURBAN:                         13
    ISOILWATER:                      14
    HYBRID_OPT:                      2
    ETAC:                            0.2

In [ ]:
bias

<xarray.DataArray 'tmmx' (time: 115, lat: 111, lon: 90)> Size: 5MB
array([[[-3.0523987 , -2.6047974 , -2.2315369 , ..., -1.9339905 ,
         -2.0104675 , -2.0960999 ],
        [-3.350891  , -2.6590881 , -2.253601  , ..., -1.777832  ,
         -1.8981628 , -2.1696472 ],
        [-3.3411865 , -2.6464539 , -2.2449646 , ..., -1.8804932 ,
         -1.9916992 , -2.258545  ],
        ...,
        [-2.2447205 , -2.0231018 , -2.047821  , ..., -4.961975  ,
         -4.9658813 , -4.967285  ],
        [-2.2452698 , -2.072174  , -2.0923462 , ..., -4.990265  ,
         -4.9888    , -5.0300903 ],
        [-2.2593384 , -2.2857666 , -2.30307   , ..., -4.980194  ,
         -5.0119934 , -5.217346  ]],

       [[-1.715332  , -1.2087708 , -0.90289307, ..., -0.78411865,
         -0.84503174, -0.83410645],
        [-1.9081421 , -1.321167  , -0.9867859 , ..., -0.44500732,
         -0.50598145, -1.0094604 ],
        [-1.8791199 , -1.3351746 , -0.99801636, ..., -0.64627075,
         -0.711853  , -1.0827637 ],
...
        [-2.1070557 , -2.0941162 , -2.0969238 , ..., -5.1730347 ,
         -5.384369  , -5.3563232 ],
        [-2.154602  , -2.1432495 , -2.1465454 , ..., -5.1926575 ,
         -5.3714905 , -5.586426  ],
        [-2.1914062 , -2.1921387 , -2.1901855 , ..., -5.219269  ,
         -5.4298096 , -5.7964783 ]],

       [[-2.2374268 , -1.9649048 , -1.6390991 , ..., -1.1194763 ,
         -1.2036743 , -1.3798828 ],
        [-2.424469  , -1.9455566 , -1.5758057 , ..., -0.96533203,
         -1.0649414 , -1.4765015 ],
        [-2.566803  , -2.0028381 , -1.6384888 , ..., -1.0585938 ,
         -1.1366272 , -1.5536194 ],
        ...,
        [-2.7431946 , -2.5399475 , -2.5273132 , ..., -5.0010986 ,
         -4.961426  , -4.996399  ],
        [-2.78833   , -2.584381  , -2.5758057 , ..., -4.9611816 ,
         -4.9916077 , -5.156952  ],
        [-2.8149414 , -2.7573853 , -2.7427368 , ..., -4.9704895 ,
         -5.0914917 , -5.284485  ]]], shape=(115, 111, 90), dtype=float32)
Coordinates:
  * time     (time) datetime64[ns] 920B 1985-04-16 1986-04-16 ... 2099-04-16
    lat      (time) float32 460B 40.78 40.78 40.78 40.78 ... 40.78 40.78 40.78
    lon      (time) float32 460B -112.0 -112.0 -112.0 ... -112.0 -112.0 -112.0
Attributes:
    FieldType:    104
    MemoryOrder:  XY 
    description:  TEMP at 2 M
    units:        K
    stagger:

In [ ]:
bias.all() == 0

<xarray.DataArray 'tmmx' ()> Size: 1B
array(True)
Attributes:
    FieldType:    104
    MemoryOrder:  XY 
    description:  TEMP at 2 M
    units:        K
    stagger:

In [ ]:
bias.any() == 0

<xarray.DataArray 'tmmx' ()> Size: 1B
array(False)
Attributes:
    FieldType:    104
    MemoryOrder:  XY 
    description:  TEMP at 2 M
    units:        K
    stagger:

In [ ]:
bias.where(bias.any()==0, drop = Ture)

NameError: name 'Ture' is not defined

In [ ]:
bias.where(bias.any()==0, drop = True)

<xarray.DataArray 'tmmx' (time: 115, lat: 111, lon: 90)> Size: 5MB
array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
...
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]]],
      shape=(115, 111, 90), dtype=float32)
Coordinates:
  * time     (time) datetime64[ns] 920B 1985-04-16 1986-04-16 ... 2099-04-16
    lat      (time) float32 460B 40.78 40.78 40.78 40.78 ... 40.78 40.78 40.78
    lon      (time) float32 460B -112.0 -112.0 -112.0 ... -112.0 -112.0 -112.0
Attributes:
    FieldType:    104
    MemoryOrder:  XY 
    description:  TEMP at 2 M
    units:        K
    stagger:

In [ ]:
bias.all()

<xarray.DataArray 'tmmx' ()> Size: 1B
array(False)
Attributes:
    FieldType:    104
    MemoryOrder:  XY 
    description:  TEMP at 2 M
    units:        K
    stagger:

In [ ]:
(bias == 0).any()

<xarray.DataArray 'tmmx' ()> Size: 1B
array(True)
Attributes:
    FieldType:    104
    MemoryOrder:  XY 
    description:  TEMP at 2 M
    units:        K
    stagger:

In [ ]:
(bias == 0).all()

<xarray.DataArray 'tmmx' ()> Size: 1B
array(False)
Attributes:
    FieldType:    104
    MemoryOrder:  XY 
    description:  TEMP at 2 M
    units:        K
    stagger:

In [ ]:
bias.where((bias == 0).any(), drop = False)

<xarray.DataArray 'tmmx' (time: 115, lat: 111, lon: 90)> Size: 5MB
array([[[-3.0523987 , -2.6047974 , -2.2315369 , ..., -1.9339905 ,
         -2.0104675 , -2.0960999 ],
        [-3.350891  , -2.6590881 , -2.253601  , ..., -1.777832  ,
         -1.8981628 , -2.1696472 ],
        [-3.3411865 , -2.6464539 , -2.2449646 , ..., -1.8804932 ,
         -1.9916992 , -2.258545  ],
        ...,
        [-2.2447205 , -2.0231018 , -2.047821  , ..., -4.961975  ,
         -4.9658813 , -4.967285  ],
        [-2.2452698 , -2.072174  , -2.0923462 , ..., -4.990265  ,
         -4.9888    , -5.0300903 ],
        [-2.2593384 , -2.2857666 , -2.30307   , ..., -4.980194  ,
         -5.0119934 , -5.217346  ]],

       [[-1.715332  , -1.2087708 , -0.90289307, ..., -0.78411865,
         -0.84503174, -0.83410645],
        [-1.9081421 , -1.321167  , -0.9867859 , ..., -0.44500732,
         -0.50598145, -1.0094604 ],
        [-1.8791199 , -1.3351746 , -0.99801636, ..., -0.64627075,
         -0.711853  , -1.0827637 ],
...
        [-2.1070557 , -2.0941162 , -2.0969238 , ..., -5.1730347 ,
         -5.384369  , -5.3563232 ],
        [-2.154602  , -2.1432495 , -2.1465454 , ..., -5.1926575 ,
         -5.3714905 , -5.586426  ],
        [-2.1914062 , -2.1921387 , -2.1901855 , ..., -5.219269  ,
         -5.4298096 , -5.7964783 ]],

       [[-2.2374268 , -1.9649048 , -1.6390991 , ..., -1.1194763 ,
         -1.2036743 , -1.3798828 ],
        [-2.424469  , -1.9455566 , -1.5758057 , ..., -0.96533203,
         -1.0649414 , -1.4765015 ],
        [-2.566803  , -2.0028381 , -1.6384888 , ..., -1.0585938 ,
         -1.1366272 , -1.5536194 ],
        ...,
        [-2.7431946 , -2.5399475 , -2.5273132 , ..., -5.0010986 ,
         -4.961426  , -4.996399  ],
        [-2.78833   , -2.584381  , -2.5758057 , ..., -4.9611816 ,
         -4.9916077 , -5.156952  ],
        [-2.8149414 , -2.7573853 , -2.7427368 , ..., -4.9704895 ,
         -5.0914917 , -5.284485  ]]], shape=(115, 111, 90), dtype=float32)
Coordinates:
  * time     (time) datetime64[ns] 920B 1985-04-16 1986-04-16 ... 2099-04-16
    lat      (time) float32 460B 40.78 40.78 40.78 40.78 ... 40.78 40.78 40.78
    lon      (time) float32 460B -112.0 -112.0 -112.0 ... -112.0 -112.0 -112.0
Attributes:
    FieldType:    104
    MemoryOrder:  XY 
    description:  TEMP at 2 M
    units:        K
    stagger:

In [ ]:
bias.where((bias == 0).any(), drop = true)

NameError: name 'true' is not defined

In [ ]:
bias.where((bias == 0).any(), drop = True)

<xarray.DataArray 'tmmx' (time: 115, lat: 111, lon: 90)> Size: 5MB
array([[[-3.0523987 , -2.6047974 , -2.2315369 , ..., -1.9339905 ,
         -2.0104675 , -2.0960999 ],
        [-3.350891  , -2.6590881 , -2.253601  , ..., -1.777832  ,
         -1.8981628 , -2.1696472 ],
        [-3.3411865 , -2.6464539 , -2.2449646 , ..., -1.8804932 ,
         -1.9916992 , -2.258545  ],
        ...,
        [-2.2447205 , -2.0231018 , -2.047821  , ..., -4.961975  ,
         -4.9658813 , -4.967285  ],
        [-2.2452698 , -2.072174  , -2.0923462 , ..., -4.990265  ,
         -4.9888    , -5.0300903 ],
        [-2.2593384 , -2.2857666 , -2.30307   , ..., -4.980194  ,
         -5.0119934 , -5.217346  ]],

       [[-1.715332  , -1.2087708 , -0.90289307, ..., -0.78411865,
         -0.84503174, -0.83410645],
        [-1.9081421 , -1.321167  , -0.9867859 , ..., -0.44500732,
         -0.50598145, -1.0094604 ],
        [-1.8791199 , -1.3351746 , -0.99801636, ..., -0.64627075,
         -0.711853  , -1.0827637 ],
...
        [-2.1070557 , -2.0941162 , -2.0969238 , ..., -5.1730347 ,
         -5.384369  , -5.3563232 ],
        [-2.154602  , -2.1432495 , -2.1465454 , ..., -5.1926575 ,
         -5.3714905 , -5.586426  ],
        [-2.1914062 , -2.1921387 , -2.1901855 , ..., -5.219269  ,
         -5.4298096 , -5.7964783 ]],

       [[-2.2374268 , -1.9649048 , -1.6390991 , ..., -1.1194763 ,
         -1.2036743 , -1.3798828 ],
        [-2.424469  , -1.9455566 , -1.5758057 , ..., -0.96533203,
         -1.0649414 , -1.4765015 ],
        [-2.566803  , -2.0028381 , -1.6384888 , ..., -1.0585938 ,
         -1.1366272 , -1.5536194 ],
        ...,
        [-2.7431946 , -2.5399475 , -2.5273132 , ..., -5.0010986 ,
         -4.961426  , -4.996399  ],
        [-2.78833   , -2.584381  , -2.5758057 , ..., -4.9611816 ,
         -4.9916077 , -5.156952  ],
        [-2.8149414 , -2.7573853 , -2.7427368 , ..., -4.9704895 ,
         -5.0914917 , -5.284485  ]]], shape=(115, 111, 90), dtype=float32)
Coordinates:
  * time     (time) datetime64[ns] 920B 1985-04-16 1986-04-16 ... 2099-04-16
    lat      (time) float32 460B 40.78 40.78 40.78 40.78 ... 40.78 40.78 40.78
    lon      (time) float32 460B -112.0 -112.0 -112.0 ... -112.0 -112.0 -112.0
Attributes:
    FieldType:    104
    MemoryOrder:  XY 
    description:  TEMP at 2 M
    units:        K
    stagger:

In [ ]:
bias.argwhere((bias == 0).any())

AttributeError: 'DataArray' object has no attribute 'argwhere'

In [ ]:
bias.where(bias == 0, drop=True)

<xarray.DataArray 'tmmx' (time: 1, lat: 1, lon: 1)> Size: 4B
array([[[0.]]], dtype=float32)
Coordinates:
  * time     (time) datetime64[ns] 8B 2056-04-16
    lat      (time) float32 4B 40.78
    lon      (time) float32 4B -112.0
Attributes:
    FieldType:    104
    MemoryOrder:  XY 
    description:  TEMP at 2 M
    units:        K
    stagger:

In [ ]:
bias == 0

<xarray.DataArray 'tmmx' (time: 115, lat: 111, lon: 90)> Size: 1MB
array([[[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
...
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]]],
      shape=(115, 111, 90))
Coordinates:
  * time     (time) datetime64[ns] 920B 1985-04-16 1986-04-16 ... 2099-04-16
    lat      (time) float32 460B 40.78 40.78 40.78 40.78 ... 40.78 40.78 40.78
    lon      (time) float32 460B -112.0 -112.0 -112.0 ... -112.0 -112.0 -112.0
Attributes:
    FieldType:    104
    MemoryOrder:  XY 
    description:  TEMP at 2 M
    units:        K
    stagger:

In [ ]:
bias.where(bias == 0, drop = True)

<xarray.DataArray 'tmmx' (time: 1, lat: 1, lon: 1)> Size: 4B
array([[[0.]]], dtype=float32)
Coordinates:
  * time     (time) datetime64[ns] 8B 2056-04-16
    lat      (time) float32 4B 40.78
    lon      (time) float32 4B -112.0
Attributes:
    FieldType:    104
    MemoryOrder:  XY 
    description:  TEMP at 2 M
    units:        K
    stagger:

In [ ]:
(bias == 0).all()

<xarray.DataArray 'tmmx' ()> Size: 1B
array(False)
Attributes:
    FieldType:    104
    MemoryOrder:  XY 
    description:  TEMP at 2 M
    units:        K
    stagger:

Connected to olympics (Python 3.11.15)

In [ ]:
"""
Author: Sydney Smith
Date Created: September 29, 2026
"""

from datetime import datetime
import gc
import glob
from loguru import logger
import numpy as np
import os
import pandas as pd
from pathlib import Path
from scipy.interpolate import interp1d
from statsmodels.distributions.empirical_distribution import ECDF, monotone_fn_inverter
import sys
import time
import xarray as xr

# ==================================
# - Establish Relative File Path - 
# ==================================

current_dir = Path(__file__).resolve().parent
sys.path.append(str(current_dir))

from plots.validation import running_window_slice, loc_sel

# ===================
# - Set Up Logger - 
# ==================

# Create directory for log files if it doesn't already exist
log_path = str(current_dir / 'log')
os.makedirs(log_path, exist_ok = True)

# String filename
log_filename = datetime.now().strftime('log/%Y-%m-%d_%H-%M-%S.log')

# Custom format for log prints
log_format = log_format = '<cyan>{time:HH:mm:ss}</cyan> | <level>{level:>8}</level> | <yellow>{name}</yellow>:<cyan>{function}</cyan>:line <magenta>{line}</magenta> - <level>{message}</level>'

# Removes the default stderr sink
logger.remove()  
# Anything above info to console
logger.add(sys.stderr, colorize = True, format = log_format, level = 'INFO') 
# Anything above debug to log file
logger.add(log_filename, colorize = False, format = log_format, level = 'DEBUG')
logger.debug(f'Log files saved to {log_path}')

# =====================
# ---- Functions ----
# =====================

def invert_ecdf(ecdf_obj):
    """
    Inverts a statsmodels ECDF object using its .y and .x attributes, regardless of if
    the values are monotonic or not, with safe extrapolation to prevent NaNs.
    """
    # ECDF from statsmodels sets first x value to -inf which causes NaNs for very small percentiles passed to the inverted function
    # Mask out any indefinite values to prevent possible NaNs
    finite_mask = np.isfinite(ecdf_obj.x)
    x_vals = ecdf_obj.x[finite_mask]
    y_vals = ecdf_obj.y[finite_mask]

    # Ensure x_vals are unique and strictly increasing for interp1d (monotonic)
    x_unique, indices = np.unique(x_vals, return_index=True)
    y_unique = y_vals[indices]

    return interp1d(
        y_unique, # Pass y values as x values
        x_unique, # And vice versa
        bounds_error = False, 
        fill_value = 'extrapolate'  # Prevents NaNs if raw percentiles push past bounds
    )

def apply_to_date(obs: xr.Dataset, hist, raw, var, date) -> np.ndarray:
    """
    Debiases data for a given date by constructing a CDF using a dataset restricted to be within a 
    running window. Date must be a valid date that exists within the dataset formatted as '%Y-%m-%d' 
    datetime object.
    """
    # TODO: check how running_window_slice handles leap years (try to set calendar year to 1988 instead since it's a leap year)

    # Slice data into 31 day running windows centered on given date (note: 31 days is defualt setting)
    obs_sliced = running_window_slice(obs, date)
    logger.info(f'NaNs in obs running window: {float(obs_sliced[var].isnull().sum().values)}')

    hist_sliced = running_window_slice(hist, date)
    logger.info(f'NaNs in hist running window: {float(hist_sliced[var].isnull().sum().values)}')

    raw_sliced = running_window_slice(raw, date)
    logger.info(f'NaNs in raw running window: {float(raw_sliced[var].isnull().sum().values)}')
    logger.info('Datasets sliced into running windows.')

    # End function if NaNs are present in the source data
    if obs_sliced[var].isnull().any() or hist_sliced[var].isnull().any() or raw_sliced[var].isnull().any():
        logger.error('NaNs detected in source data.')
        return NotImplementedError

    # Create CDFs from sliced datasets
    obs_cdf = ECDF(obs_sliced[var].values.flatten())
    hist_cdf = ECDF(hist_sliced[var].values.flatten())
    raw_cdf = ECDF(raw_sliced[var].values.flatten())

    # Select day of interest and create mask
    day = int(date.strftime('%d'))
    month = int(date.strftime('%m'))
    mask = (raw.time.dt.month == month) & (raw.time.dt.day == day)

    # Skip dates that don't exist
    if not mask.any():
        logger.warning(f'No date found for {date}.')
        return None
    
    # Apply mask to dataset
    raw_date = raw.where(mask, drop = True) # This should select day of interest across all years in ds
    logger.info(f'Dataset sliced down to date of interest: {date}')
    logger.info(raw_date)

    # Use raw_cdf like a function to find all of the percentiles in raw_date
    raw_percentiles = raw_cdf(raw_date[var].values)
    logger.info(f'Percentile values for raw data: {raw_percentiles}')
    logger.info(f'NaNs in raw_percentiles: {float(np.isnan(raw_percentiles).sum())}')

    # TODO: NaNs are appearing in obs and hist from this point on
    # Invert hist_cdf and obs_cdf to take a percentile and output a data point
    inv_obs = invert_ecdf(obs_cdf)
    inv_hist = invert_ecdf(hist_cdf)

    # Find data points for every percentile in raw_percentiles
    obs_vals = inv_obs(raw_percentiles)
    logger.info(f'NaNs in obs_vals: {float(np.isnan(obs_vals).sum())}')
    logger.info(f'Obs values for given percentiles: {obs_vals}')

    hist_vals = inv_hist(raw_percentiles)
    logger.info(f'NaNs in hist_vals: {float(np.isnan(hist_vals).sum())}')
    logger.info(f'Historical values for given percentiles: {hist_vals}')

    # Perform bias adjustment
    bias = hist_vals - obs_vals
    logger.info(f'NaNs in bias: {float(np.isnan(bias).sum())}')
    raw_bias_corrected = raw_date[var].values - bias

    # End script early if there are Nans in data
    if np.isnan(raw_bias_corrected).any():
        logger.error(f'NaNs in debiased data: {float(np.isnan(raw_bias_corrected).sum())}')
        logger.info(f'Nan error occured on {date}.')
        return None

    logger.info('Raw data successfully bias corrected!')
    logger.info(raw_bias_corrected)
    logger.info(f'Bias corrected max: {raw_bias_corrected.max()}')
    logger.info(f'Bias corrected min: {raw_bias_corrected.min()}')

    # Close out of unwanted datasets
    obs_sliced.close()
    hist_sliced.close()
    raw_sliced.close()
    raw_date.close()
    del obs_vals
    del hist_vals
    del raw_percentiles
    gc.collect()

    return raw_bias_corrected

def ECDFM(obs, raw, var, lat, lon):
    """
    Performs ECDFM quantile mapping debiasing process on given data based on data from a given histroical
    period and corresponding observational data. Raw data should include both historical and future data. 
    """

    # Slice raw data down to historical period
    hist = raw.sel(time = slice('1985-01-01', '2014-12-31'))
    
    # Select specific location of interest (choses nearest possible location)
    obs_loc = loc_sel(obs, lat = lat, lon = lon)
    logger.info(f'NaNs at selected obs location: {float(obs_loc[var].isnull().sum().values)}')
    hist_loc = loc_sel(hist, lat = lat, lon = lon)
    logger.info(f'NaNs at selected hist location: {float(hist_loc[var].isnull().sum().values)}')
    raw_loc = loc_sel(raw, lat = lat, lon = lon)
    logger.info(f'NaNs at selected raw location: {float(raw_loc[var].isnull().sum().values)}')
    logger.info(raw_loc[var])

    # Make copy of raw_loc to plug debiased data into
    template = raw_loc.copy()

    # Create date range that spans a calendar year
    dates = pd.date_range(start = '1985-01-01', end = '1985-12-31', freq = 'D') 

    for date in dates:
        # Apply bias correction for single date
        logger.info(f'Date type: {type(date)}')
        debiased_doy = apply_to_date(obs_loc, hist_loc, raw_loc, var, date)

        # Exit out of function if Nans occur
        if debiased_doy is None:
            logger.error(f'Error detected in dataset. See log above for more information.')
            return None

        # Pull month and day from date
        day = int(date.strftime('%d'))
        month = int(date.strftime('%m'))

        # Plug in debiased data
        mask = (template.time.dt.month == month) & (template.time.dt.day == day)
        template[var].data[mask] = debiased_doy 
        logger.info(f'Debiased data for {date} added to dataset!')

        # Back into data's bias to ensure that the bias corrected data doesn't equal to raw data
        temp_masked = template[var].where(mask, drop = True)
        raw_masked = raw[var].where(mask, drop = True)
        bias = raw_masked - temp_masked # Back into data's bias
        logger.info(f'bias type: {type(bias)}')

        # Ensure that bias is not all zeros
        if (bias == 0).all(): # Ensure it's not all zeros
            logger.error('Bias corrected data was not successfully applied to dataset.')
            return None
        else:
            logger.info(f'Bias for {date} successfully corrected.')

        # Close unnecessay datasets
        temp_masked.close()
        raw_masked.close()

    return template

def nan_test(var):
    """
    Returns location of Nans in debiased data.
    """
    
    fpath = glob.glob(str(current_dir / 'wrfout' / f'*{var}*.nc'))
    debiased = xr.open_dataset(fpath[0], decode_times = True)

    nan_data = debiased[var].where(debiased[var].isnull(), drop = True)

    return nan_data

def np_nan_loc(arr):
    """
    Find locations of NaNs in np.array.
    """
    return np.argwhere(np.isnan(arr))

# Catch silent errors and report to log file
@logger.catch 
def main(var, data_location, save = True):
    # Open datasets
    obs_path = glob.glob(str(data_location / 'gridMET' / f'*{var}*.nc'))
    obs = xr.open_dataset(obs_path[0], decode_times = True)

    raw_path = glob.glob(str(data_location / 'daily' / f'*{var}*.nc'))
    raw = xr.open_dataset(raw_path[0], decode_times = True)

    # Set location of interest as SLC airport
    lat = 40.788
    lon = -111.978

    # Pass datasets to debiaser
    test = ECDFM(obs, raw, var, lat, lon)

    if save:
        # Create output directory to store new cleaned files
        output_dir = current_dir / 'wrfout' 
        os.makedirs(output_dir, exist_ok = True) # Don't make if it already exists

        # Generate save name for debiased data
        out_path = os.path.join(output_dir, f'wrfout_GSLBIP_multimodel_ssp245_{var}_test.nc')

        # Save data to netCDF file
        test.to_netcdf(out_path)
        logger.success(f'File saved to: {out_path}')

    # Close out of data
    test.close()

# if __name__ == '__main__':

#     # Track program time in log files
#     start = time.perf_counter()
#     logger.info('Beginning execution.')

#     # Only inputs required
#     main(
#             var = 'tmmx',
#             data_location = current_dir,
#             save = True
#             )

#     # Report of runtime at completion 
#     logger.success(f'Debiasing process completed!')
#     logger.info(f'Total runtime: {time.perf_counter() - start:.4f}s')

#     # Force script to stop running once code is finished
#     sys.exit(0)


var = 'tmmx'
# Open datasets
obs_path = glob.glob(str(current_dir / 'gridMET' / f'*{var}*.nc'))
obs = xr.open_dataset(obs_path[0], decode_times = True)

raw_path = glob.glob(str(current_dir / 'daily' / f'*{var}*.nc'))
raw = xr.open_dataset(raw_path[0], decode_times = True)

# Set location of interest as SLC airport
lat = 40.788
lon = -111.978


# Slice raw data down to historical period
hist = raw.sel(time = slice('1985-01-01', '2014-12-31'))

# Select specific location of interest (choses nearest possible location)
obs_loc = loc_sel(obs, lat = lat, lon = lon)
logger.info(f'NaNs at selected obs location: {float(obs_loc[var].isnull().sum().values)}')
hist_loc = loc_sel(hist, lat = lat, lon = lon)
logger.info(f'NaNs at selected hist location: {float(hist_loc[var].isnull().sum().values)}')
raw_loc = loc_sel(raw, lat = lat, lon = lon)
logger.info(f'NaNs at selected raw location: {float(raw_loc[var].isnull().sum().values)}')
logger.info(raw_loc[var])

# Make copy of raw_loc to plug debiased data into
template = raw_loc.copy()

# Create date range that spans a calendar year
date = pd.to_datetime('1985-12-31') 

# Apply bias correction for single date
logger.info(f'Date type: {type(date)}')
debiased_doy = apply_to_date(obs_loc, hist_loc, raw_loc, var, date)

# Exit out of function if Nans occur
if debiased_doy is None:
    logger.error(f'Nans detected in dataset. See log above for more information.')

# Pull month and day from date
day = int(date.strftime('%d'))
month = int(date.strftime('%m'))

# Plug in debiased data
mask = (template.time.dt.month == month) & (template.time.dt.day == day)
template[var].data[mask] = debiased_doy 
logger.info(f'Debiased data for {date} added to dataset!')

# Check that data was actaully replaced
temp_masked = template[var].where(mask, drop = True)
raw_masked = raw[var].where(mask, drop = True)
bias = raw_masked - temp_masked # Back into data's bias
if (bias.all() == 0): # Ensure it's not all zeros
    logger.error('Bias corrected data was not successfully applied to dataset.')
else:
    logger.info(f'Bias for {date} successfully corrected.')

18:19:54 |     INFO | plots.validation:loc_sel:line 89 - Selecting nearest location to 40.788, -111.978.
/uufs/chpc.utah.edu/common/home/strong-group7/sydney/miniforge3_envs/olympics/lib/python3.11/site-packages/xarray/core/dataarray.py:6318: FutureWarning: Behaviour of argmin/argmax with neither dim nor axis argument will change to return a dict of indices of each dimension. To get a single, flat index, please use np.argmin(da.data) or np.argmax(da.data) instead of da.argmin() or da.argmax().
  result = self.variable.argmin(dim, axis, keep_attrs, skipna)
18:19:54 |     INFO | plots.validation:loc_sel:line 100 - Point selected is <xarray.DataArray ()> Size: 8B
array(3916)
Coordinates:
    time     datetime64[ns] 8B 1985-01-01 from 40.788, -111.978.
18:19:54 |     INFO | plots.validation:loc_sel:line 102 - Min distance is located at 46, 43.
18:19:54 |     INFO | plots.validation:loc_sel:line 113 - New lat: 40.78413391113281
18:19:54 |     INFO | plots.validation:loc_sel:line 114 - New l

ValueError: zip() argument 2 is longer than argument 1

In [ ]:
day

31

In [ ]:
month

12

In [ ]:
mask

<xarray.DataArray (time: 41888)> Size: 42kB
array([False, False, False, ..., False, False, False], shape=(41888,))
Coordinates:
  * time     (time) datetime64[ns] 335kB 1985-01-01 1985-01-02 ... 2099-12-30
    lat      (time) float32 168kB 40.78 40.78 40.78 40.78 ... 40.78 40.78 40.78
    lon      (time) float32 168kB -112.0 -112.0 -112.0 ... -112.0 -112.0 -112.0

In [ ]:
temp_masked

NameError: name 'temp_masked' is not defined

In [ ]:
"""
Author: Sydney Smith
Date Created: September 29, 2026
"""

from datetime import datetime
import gc
import glob
from loguru import logger
import numpy as np
import os
import pandas as pd
from pathlib import Path
from scipy.interpolate import interp1d
from statsmodels.distributions.empirical_distribution import ECDF, monotone_fn_inverter
import sys
import time
import xarray as xr

# ==================================
# - Establish Relative File Path - 
# ==================================

current_dir = Path(__file__).resolve().parent
sys.path.append(str(current_dir))

from plots.validation import running_window_slice, loc_sel

# ===================
# - Set Up Logger - 
# ==================

# Create directory for log files if it doesn't already exist
log_path = str(current_dir / 'log')
os.makedirs(log_path, exist_ok = True)

# String filename
log_filename = datetime.now().strftime('log/%Y-%m-%d_%H-%M-%S.log')

# Custom format for log prints
log_format = log_format = '<cyan>{time:HH:mm:ss}</cyan> | <level>{level:>8}</level> | <yellow>{name}</yellow>:<cyan>{function}</cyan>:line <magenta>{line}</magenta> - <level>{message}</level>'

# Removes the default stderr sink
logger.remove()  
# Anything above info to console
logger.add(sys.stderr, colorize = True, format = log_format, level = 'INFO') 
# Anything above debug to log file
logger.add(log_filename, colorize = False, format = log_format, level = 'DEBUG')
logger.debug(f'Log files saved to {log_path}')

# =====================
# ---- Functions ----
# =====================

def invert_ecdf(ecdf_obj):
    """
    Inverts a statsmodels ECDF object using its .y and .x attributes, regardless of if
    the values are monotonic or not, with safe extrapolation to prevent NaNs.
    """
    # ECDF from statsmodels sets first x value to -inf which causes NaNs for very small percentiles passed to the inverted function
    # Mask out any indefinite values to prevent possible NaNs
    finite_mask = np.isfinite(ecdf_obj.x)
    x_vals = ecdf_obj.x[finite_mask]
    y_vals = ecdf_obj.y[finite_mask]

    # Ensure x_vals are unique and strictly increasing for interp1d (monotonic)
    x_unique, indices = np.unique(x_vals, return_index=True)
    y_unique = y_vals[indices]

    return interp1d(
        y_unique, # Pass y values as x values
        x_unique, # And vice versa
        bounds_error = False, 
        fill_value = 'extrapolate'  # Prevents NaNs if raw percentiles push past bounds
    )

def apply_to_date(obs: xr.Dataset, hist, raw, var, date) -> np.ndarray:
    """
    Debiases data for a given date by constructing a CDF using a dataset restricted to be within a 
    running window. Date must be a valid date that exists within the dataset formatted as '%Y-%m-%d' 
    datetime object.
    """
    # TODO: check how running_window_slice handles leap years (try to set calendar year to 1988 instead since it's a leap year)

    # Slice data into 31 day running windows centered on given date (note: 31 days is defualt setting)
    obs_sliced = running_window_slice(obs, date)
    logger.info(f'NaNs in obs running window: {float(obs_sliced[var].isnull().sum().values)}')

    hist_sliced = running_window_slice(hist, date)
    logger.info(f'NaNs in hist running window: {float(hist_sliced[var].isnull().sum().values)}')

    raw_sliced = running_window_slice(raw, date)
    logger.info(f'NaNs in raw running window: {float(raw_sliced[var].isnull().sum().values)}')
    logger.info('Datasets sliced into running windows.')

    # End function if NaNs are present in the source data
    if obs_sliced[var].isnull().any() or hist_sliced[var].isnull().any() or raw_sliced[var].isnull().any():
        logger.error('NaNs detected in source data.')
        return NotImplementedError

    # Create CDFs from sliced datasets
    obs_cdf = ECDF(obs_sliced[var].values.flatten())
    hist_cdf = ECDF(hist_sliced[var].values.flatten())
    raw_cdf = ECDF(raw_sliced[var].values.flatten())

    # Select day of interest and create mask
    day = int(date.strftime('%d'))
    month = int(date.strftime('%m'))
    mask = (raw.time.dt.month == month) & (raw.time.dt.day == day)

    # Skip dates that don't exist
    if not mask.any():
        logger.warning(f'No date found for {date}.')
        return None
    
    # Apply mask to dataset
    raw_date = raw.where(mask, drop = True) # This should select day of interest across all years in ds
    logger.info(f'Dataset sliced down to date of interest: {date}')
    logger.info(raw_date)

    # Use raw_cdf like a function to find all of the percentiles in raw_date
    raw_percentiles = raw_cdf(raw_date[var].values)
    logger.info(f'Percentile values for raw data: {raw_percentiles}')
    logger.info(f'NaNs in raw_percentiles: {float(np.isnan(raw_percentiles).sum())}')

    # TODO: NaNs are appearing in obs and hist from this point on
    # Invert hist_cdf and obs_cdf to take a percentile and output a data point
    inv_obs = invert_ecdf(obs_cdf)
    inv_hist = invert_ecdf(hist_cdf)

    # Find data points for every percentile in raw_percentiles
    obs_vals = inv_obs(raw_percentiles)
    logger.info(f'NaNs in obs_vals: {float(np.isnan(obs_vals).sum())}')
    logger.info(f'Obs values for given percentiles: {obs_vals}')

    hist_vals = inv_hist(raw_percentiles)
    logger.info(f'NaNs in hist_vals: {float(np.isnan(hist_vals).sum())}')
    logger.info(f'Historical values for given percentiles: {hist_vals}')

    # Perform bias adjustment
    bias = hist_vals - obs_vals
    logger.info(f'NaNs in bias: {float(np.isnan(bias).sum())}')
    raw_bias_corrected = raw_date[var].values - bias

    # End script early if there are Nans in data
    if np.isnan(raw_bias_corrected).any():
        logger.error(f'NaNs in debiased data: {float(np.isnan(raw_bias_corrected).sum())}')
        logger.info(f'Nan error occured on {date}.')
        return None

    logger.info('Raw data successfully bias corrected!')
    logger.info(raw_bias_corrected)
    logger.info(f'Bias corrected max: {raw_bias_corrected.max()}')
    logger.info(f'Bias corrected min: {raw_bias_corrected.min()}')

    # Close out of unwanted datasets
    obs_sliced.close()
    hist_sliced.close()
    raw_sliced.close()
    raw_date.close()
    del obs_vals
    del hist_vals
    del raw_percentiles
    gc.collect()

    return raw_bias_corrected

def ECDFM(obs, raw, var, lat, lon):
    """
    Performs ECDFM quantile mapping debiasing process on given data based on data from a given histroical
    period and corresponding observational data. Raw data should include both historical and future data. 
    """

    # Slice raw data down to historical period
    hist = raw.sel(time = slice('1985-01-01', '2014-12-31'))
    
    # Select specific location of interest (choses nearest possible location)
    obs_loc = loc_sel(obs, lat = lat, lon = lon)
    logger.info(f'NaNs at selected obs location: {float(obs_loc[var].isnull().sum().values)}')
    hist_loc = loc_sel(hist, lat = lat, lon = lon)
    logger.info(f'NaNs at selected hist location: {float(hist_loc[var].isnull().sum().values)}')
    raw_loc = loc_sel(raw, lat = lat, lon = lon)
    logger.info(f'NaNs at selected raw location: {float(raw_loc[var].isnull().sum().values)}')
    logger.info(raw_loc[var])

    # Make copy of raw_loc to plug debiased data into
    template = raw_loc.copy()

    # Create date range that spans a calendar year
    dates = pd.date_range(start = '1985-01-01', end = '1985-12-31', freq = 'D') 

    for date in dates:
        # Apply bias correction for single date
        logger.info(f'Date type: {type(date)}')
        debiased_doy = apply_to_date(obs_loc, hist_loc, raw_loc, var, date)

        # Exit out of function if Nans occur
        if debiased_doy is None:
            logger.error(f'Error detected in dataset. See log above for more information.')
            return None

        # Pull month and day from date
        day = int(date.strftime('%d'))
        month = int(date.strftime('%m'))

        # Plug in debiased data
        mask = (template.time.dt.month == month) & (template.time.dt.day == day)
        template[var].data[mask] = debiased_doy 
        logger.info(f'Debiased data for {date} added to dataset!')

        # Back into data's bias to ensure that the bias corrected data doesn't equal to raw data
        temp_masked = template[var].where(mask, drop = True)
        raw_masked = raw[var].where(mask, drop = True)
        bias = raw_masked - temp_masked # Back into data's bias
        logger.info(f'bias type: {type(bias)}')

        # Ensure that bias is not all zeros
        if (bias == 0).all(): # Ensure it's not all zeros
            logger.error('Bias corrected data was not successfully applied to dataset.')
            return None
        else:
            logger.info(f'Bias for {date} successfully corrected.')

        # Close unnecessay datasets
        temp_masked.close()
        raw_masked.close()

    return template

def nan_test(var):
    """
    Returns location of Nans in debiased data.
    """
    
    fpath = glob.glob(str(current_dir / 'wrfout' / f'*{var}*.nc'))
    debiased = xr.open_dataset(fpath[0], decode_times = True)

    nan_data = debiased[var].where(debiased[var].isnull(), drop = True)

    return nan_data

def np_nan_loc(arr):
    """
    Find locations of NaNs in np.array.
    """
    return np.argwhere(np.isnan(arr))

# Catch silent errors and report to log file
@logger.catch 
def main(var, data_location, save = True):
    # Open datasets
    obs_path = glob.glob(str(data_location / 'gridMET' / f'*{var}*.nc'))
    obs = xr.open_dataset(obs_path[0], decode_times = True)

    raw_path = glob.glob(str(data_location / 'daily' / f'*{var}*.nc'))
    raw = xr.open_dataset(raw_path[0], decode_times = True)

    # Set location of interest as SLC airport
    lat = 40.788
    lon = -111.978

    # Pass datasets to debiaser
    test = ECDFM(obs, raw, var, lat, lon)

    if save:
        # Create output directory to store new cleaned files
        output_dir = current_dir / 'wrfout' 
        os.makedirs(output_dir, exist_ok = True) # Don't make if it already exists

        # Generate save name for debiased data
        out_path = os.path.join(output_dir, f'wrfout_GSLBIP_multimodel_ssp245_{var}_test.nc')

        # Save data to netCDF file
        test.to_netcdf(out_path)
        logger.success(f'File saved to: {out_path}')

    # Close out of data
    test.close()

# if __name__ == '__main__':

#     # Track program time in log files
#     start = time.perf_counter()
#     logger.info('Beginning execution.')

#     # Only inputs required
#     main(
#             var = 'tmmx',
#             data_location = current_dir,
#             save = True
#             )

#     # Report of runtime at completion 
#     logger.success(f'Debiasing process completed!')
#     logger.info(f'Total runtime: {time.perf_counter() - start:.4f}s')

#     # Force script to stop running once code is finished
#     sys.exit(0)


var = 'tmmx'
# Open datasets
obs_path = glob.glob(str(current_dir / 'gridMET' / f'*{var}*.nc'))
obs = xr.open_dataset(obs_path[0], decode_times = True)

raw_path = glob.glob(str(current_dir / 'daily' / f'*{var}*.nc'))
raw = xr.open_dataset(raw_path[0], decode_times = True)

# Set location of interest as SLC airport
lat = 40.788
lon = -111.978


# Slice raw data down to historical period
hist = raw.sel(time = slice('1985-01-01', '2014-12-31'))

# Select specific location of interest (choses nearest possible location)
obs_loc = loc_sel(obs, lat = lat, lon = lon)
logger.info(f'NaNs at selected obs location: {float(obs_loc[var].isnull().sum().values)}')
hist_loc = loc_sel(hist, lat = lat, lon = lon)
logger.info(f'NaNs at selected hist location: {float(hist_loc[var].isnull().sum().values)}')
raw_loc = loc_sel(raw, lat = lat, lon = lon)
logger.info(f'NaNs at selected raw location: {float(raw_loc[var].isnull().sum().values)}')
logger.info(raw_loc[var])

# Make copy of raw_loc to plug debiased data into
template = raw_loc.copy()

# Create date range that spans a calendar year
date = pd.to_datetime('1985-12-31') 

# Apply bias correction for single date
logger.info(f'Date type: {type(date)}')
# debiased_doy = apply_to_date(obs_loc, hist_loc, raw_loc, var, date)

# Slice data into 31 day running windows centered on given date (note: 31 days is defualt setting)
obs_sliced = running_window_slice(obs, date)
logger.info(f'NaNs in obs running window: {float(obs_sliced[var].isnull().sum().values)}')

hist_sliced = running_window_slice(hist, date)
logger.info(f'NaNs in hist running window: {float(hist_sliced[var].isnull().sum().values)}')

raw_sliced = running_window_slice(raw, date)
logger.info(f'NaNs in raw running window: {float(raw_sliced[var].isnull().sum().values)}')
logger.info('Datasets sliced into running windows.')

# End function if NaNs are present in the source data
if obs_sliced[var].isnull().any() or hist_sliced[var].isnull().any() or raw_sliced[var].isnull().any():
    logger.error('NaNs detected in source data.')

# Create CDFs from sliced datasets
obs_cdf = ECDF(obs_sliced[var].values.flatten())
hist_cdf = ECDF(hist_sliced[var].values.flatten())
raw_cdf = ECDF(raw_sliced[var].values.flatten())

# Select day of interest and create mask
day = int(date.strftime('%d'))
month = int(date.strftime('%m'))
mask = (raw.time.dt.month == month) & (raw.time.dt.day == day)

# Skip dates that don't exist
if not mask.any():
    logger.warning(f'No date found for {date}.')

# Apply mask to dataset
raw_date = raw.where(mask, drop = True) # This should select day of interest across all years in ds
logger.info(f'Dataset sliced down to date of interest: {date}')
logger.info(raw_date)

18:24:35 |     INFO | plots.validation:loc_sel:line 89 - Selecting nearest location to 40.788, -111.978.
/uufs/chpc.utah.edu/common/home/strong-group7/sydney/miniforge3_envs/olympics/lib/python3.11/site-packages/xarray/core/dataarray.py:6318: FutureWarning: Behaviour of argmin/argmax with neither dim nor axis argument will change to return a dict of indices of each dimension. To get a single, flat index, please use np.argmin(da.data) or np.argmax(da.data) instead of da.argmin() or da.argmax().
  result = self.variable.argmin(dim, axis, keep_attrs, skipna)
18:24:35 |     INFO | plots.validation:loc_sel:line 100 - Point selected is <xarray.DataArray ()> Size: 8B
array(3916)
Coordinates:
    time     datetime64[ns] 8B 1985-01-01 from 40.788, -111.978.
18:24:35 |     INFO | plots.validation:loc_sel:line 102 - Min distance is located at 46, 43.
18:24:35 |     INFO | plots.validation:loc_sel:line 113 - New lat: 40.78413391113281
18:24:35 |     INFO | plots.validation:loc_sel:line 114 - New l

ValueError: size of dimension 'lat' on inputs was unexpectedly changed by applied function from 111 to 1. Only dimensions specified in ``exclude_dims`` with xarray.apply_ufunc are allowed to change size. The data returned was:

array([], shape=(0, 1, 1), dtype=float32)

In [ ]:
mask

<xarray.DataArray (time: 41888)> Size: 42kB
array([False, False, False, ..., False, False, False], shape=(41888,))
Coordinates:
  * time     (time) datetime64[ns] 335kB 1985-01-01 1985-01-02 ... 2099-12-30

In [ ]:
day

31

In [ ]:
month

12

In [ ]:
raw.sel(time = (raw.time.dt.month == month) & (raw.time.dt.day == day))

<xarray.Dataset> Size: 0B
Dimensions:  (time: 0, lat: 111, lon: 90, south_north: 111, west_east: 90)
Coordinates:
  * time     (time) datetime64[ns] 0B 
    lat      (time, south_north, west_east) float32 0B ...
    lon      (time, south_north, west_east) float32 0B ...
Dimensions without coordinates: south_north, west_east
Data variables:
    tmmx     (time, lat, lon) float32 0B ...
Attributes: (12/155)
    TITLE:                            OUTPUT FROM WRF V4.6.0 MODEL
    START_DATE:                      1984-01-01_06:00:00
    SIMULATION_START_DATE:           1984-01-01_06:00:00
    WEST-EAST_GRID_DIMENSION:        91
    SOUTH-NORTH_GRID_DIMENSION:      112
    BOTTOM-TOP_GRID_DIMENSION:       39
    ...                              ...
    ISLAKE:                          21
    ISICE:                           15
    ISURBAN:                         13
    ISOILWATER:                      14
    HYBRID_OPT:                      2
    ETAC:                            0.2

In [ ]:
raw.sel(time = (raw.time.dt.month == 11) & (raw.time.dt.day == day))

<xarray.Dataset> Size: 0B
Dimensions:  (time: 0, lat: 111, lon: 90, south_north: 111, west_east: 90)
Coordinates:
  * time     (time) datetime64[ns] 0B 
    lat      (time, south_north, west_east) float32 0B ...
    lon      (time, south_north, west_east) float32 0B ...
Dimensions without coordinates: south_north, west_east
Data variables:
    tmmx     (time, lat, lon) float32 0B ...
Attributes: (12/155)
    TITLE:                            OUTPUT FROM WRF V4.6.0 MODEL
    START_DATE:                      1984-01-01_06:00:00
    SIMULATION_START_DATE:           1984-01-01_06:00:00
    WEST-EAST_GRID_DIMENSION:        91
    SOUTH-NORTH_GRID_DIMENSION:      112
    BOTTOM-TOP_GRID_DIMENSION:       39
    ...                              ...
    ISLAKE:                          21
    ISICE:                           15
    ISURBAN:                         13
    ISOILWATER:                      14
    HYBRID_OPT:                      2
    ETAC:                            0.2

In [ ]:
"""
Author: Sydney Smith
Date Created: September 29, 2026
"""

from datetime import datetime
import gc
import glob
from loguru import logger
import numpy as np
import os
import pandas as pd
from pathlib import Path
from scipy.interpolate import interp1d
from statsmodels.distributions.empirical_distribution import ECDF, monotone_fn_inverter
import sys
import time
import xarray as xr

# ==================================
# - Establish Relative File Path - 
# ==================================

current_dir = Path(__file__).resolve().parent
sys.path.append(str(current_dir))

from plots.validation import running_window_slice, loc_sel

# ===================
# - Set Up Logger - 
# ==================

# Create directory for log files if it doesn't already exist
log_path = str(current_dir / 'log')
os.makedirs(log_path, exist_ok = True)

# String filename
log_filename = datetime.now().strftime('log/%Y-%m-%d_%H-%M-%S.log')

# Custom format for log prints
log_format = log_format = '<cyan>{time:HH:mm:ss}</cyan> | <level>{level:>8}</level> | <yellow>{name}</yellow>:<cyan>{function}</cyan>:line <magenta>{line}</magenta> - <level>{message}</level>'

# Removes the default stderr sink
logger.remove()  
# Anything above info to console
logger.add(sys.stderr, colorize = True, format = log_format, level = 'INFO') 
# Anything above debug to log file
logger.add(log_filename, colorize = False, format = log_format, level = 'DEBUG')
logger.debug(f'Log files saved to {log_path}')

# =====================
# ---- Functions ----
# =====================

def invert_ecdf(ecdf_obj):
    """
    Inverts a statsmodels ECDF object using its .y and .x attributes, regardless of if
    the values are monotonic or not, with safe extrapolation to prevent NaNs.
    """
    # ECDF from statsmodels sets first x value to -inf which causes NaNs for very small percentiles passed to the inverted function
    # Mask out any indefinite values to prevent possible NaNs
    finite_mask = np.isfinite(ecdf_obj.x)
    x_vals = ecdf_obj.x[finite_mask]
    y_vals = ecdf_obj.y[finite_mask]

    # Ensure x_vals are unique and strictly increasing for interp1d (monotonic)
    x_unique, indices = np.unique(x_vals, return_index=True)
    y_unique = y_vals[indices]

    return interp1d(
        y_unique, # Pass y values as x values
        x_unique, # And vice versa
        bounds_error = False, 
        fill_value = 'extrapolate'  # Prevents NaNs if raw percentiles push past bounds
    )

def apply_to_date(obs: xr.Dataset, hist, raw, var, date) -> np.ndarray:
    """
    Debiases data for a given date by constructing a CDF using a dataset restricted to be within a 
    running window. Date must be a valid date that exists within the dataset formatted as '%Y-%m-%d' 
    datetime object.
    """
    # TODO: check how running_window_slice handles leap years (try to set calendar year to 1988 instead since it's a leap year)

    # Slice data into 31 day running windows centered on given date (note: 31 days is defualt setting)
    obs_sliced = running_window_slice(obs, date)
    logger.info(f'NaNs in obs running window: {float(obs_sliced[var].isnull().sum().values)}')

    hist_sliced = running_window_slice(hist, date)
    logger.info(f'NaNs in hist running window: {float(hist_sliced[var].isnull().sum().values)}')

    raw_sliced = running_window_slice(raw, date)
    logger.info(f'NaNs in raw running window: {float(raw_sliced[var].isnull().sum().values)}')
    logger.info('Datasets sliced into running windows.')

    # End function if NaNs are present in the source data
    if obs_sliced[var].isnull().any() or hist_sliced[var].isnull().any() or raw_sliced[var].isnull().any():
        logger.error('NaNs detected in source data.')
        return NotImplementedError

    # Create CDFs from sliced datasets
    obs_cdf = ECDF(obs_sliced[var].values.flatten())
    hist_cdf = ECDF(hist_sliced[var].values.flatten())
    raw_cdf = ECDF(raw_sliced[var].values.flatten())

    # Select day of interest and create mask
    day = int(date.strftime('%d'))
    month = int(date.strftime('%m'))
    mask = (raw.time.dt.month == month) & (raw.time.dt.day == day)

    # Skip dates that don't exist
    if not mask.any():
        logger.warning(f'No date found for {date}.')
        return None
    
    # Apply mask to dataset
    raw_date = raw.where(mask, drop = True) # This should select day of interest across all years in ds
    logger.info(f'Dataset sliced down to date of interest: {date}')
    logger.info(raw_date)

    # Use raw_cdf like a function to find all of the percentiles in raw_date
    raw_percentiles = raw_cdf(raw_date[var].values)
    logger.info(f'Percentile values for raw data: {raw_percentiles}')
    logger.info(f'NaNs in raw_percentiles: {float(np.isnan(raw_percentiles).sum())}')

    # TODO: NaNs are appearing in obs and hist from this point on
    # Invert hist_cdf and obs_cdf to take a percentile and output a data point
    inv_obs = invert_ecdf(obs_cdf)
    inv_hist = invert_ecdf(hist_cdf)

    # Find data points for every percentile in raw_percentiles
    obs_vals = inv_obs(raw_percentiles)
    logger.info(f'NaNs in obs_vals: {float(np.isnan(obs_vals).sum())}')
    logger.info(f'Obs values for given percentiles: {obs_vals}')

    hist_vals = inv_hist(raw_percentiles)
    logger.info(f'NaNs in hist_vals: {float(np.isnan(hist_vals).sum())}')
    logger.info(f'Historical values for given percentiles: {hist_vals}')

    # Perform bias adjustment
    bias = hist_vals - obs_vals
    logger.info(f'NaNs in bias: {float(np.isnan(bias).sum())}')
    raw_bias_corrected = raw_date[var].values - bias

    # End script early if there are Nans in data
    if np.isnan(raw_bias_corrected).any():
        logger.error(f'NaNs in debiased data: {float(np.isnan(raw_bias_corrected).sum())}')
        logger.info(f'Nan error occured on {date}.')
        return None

    logger.info('Raw data successfully bias corrected!')
    logger.info(raw_bias_corrected)
    logger.info(f'Bias corrected max: {raw_bias_corrected.max()}')
    logger.info(f'Bias corrected min: {raw_bias_corrected.min()}')

    # Close out of unwanted datasets
    obs_sliced.close()
    hist_sliced.close()
    raw_sliced.close()
    raw_date.close()
    del obs_vals
    del hist_vals
    del raw_percentiles
    gc.collect()

    return raw_bias_corrected

def ECDFM(obs, raw, var, lat, lon):
    """
    Performs ECDFM quantile mapping debiasing process on given data based on data from a given histroical
    period and corresponding observational data. Raw data should include both historical and future data. 
    """

    # Slice raw data down to historical period
    hist = raw.sel(time = slice('1985-01-01', '2014-12-31'))
    
    # Select specific location of interest (choses nearest possible location)
    obs_loc = loc_sel(obs, lat = lat, lon = lon)
    logger.info(f'NaNs at selected obs location: {float(obs_loc[var].isnull().sum().values)}')
    hist_loc = loc_sel(hist, lat = lat, lon = lon)
    logger.info(f'NaNs at selected hist location: {float(hist_loc[var].isnull().sum().values)}')
    raw_loc = loc_sel(raw, lat = lat, lon = lon)
    logger.info(f'NaNs at selected raw location: {float(raw_loc[var].isnull().sum().values)}')
    logger.info(raw_loc[var])

    # Make copy of raw_loc to plug debiased data into
    template = raw_loc.copy()

    # Create date range that spans a calendar year
    dates = pd.date_range(start = '1985-01-01', end = '1985-12-31', freq = 'D') 

    for date in dates:
        # Apply bias correction for single date
        logger.info(f'Date type: {type(date)}')
        debiased_doy = apply_to_date(obs_loc, hist_loc, raw_loc, var, date)

        # Exit out of function if Nans occur
        if debiased_doy is None:
            logger.error(f'Error detected in dataset. See log above for more information.')
            return None

        # Pull month and day from date
        day = int(date.strftime('%d'))
        month = int(date.strftime('%m'))

        # Plug in debiased data
        mask = (template.time.dt.month == month) & (template.time.dt.day == day)
        template[var].data[mask] = debiased_doy 
        logger.info(f'Debiased data for {date} added to dataset!')

        # Back into data's bias to ensure that the bias corrected data doesn't equal to raw data
        temp_masked = template[var].where(mask, drop = True)
        raw_masked = raw[var].where(mask, drop = True)
        bias = raw_masked - temp_masked # Back into data's bias
        logger.info(f'bias type: {type(bias)}')

        # Ensure that bias is not all zeros
        if (bias == 0).all(): # Ensure it's not all zeros
            logger.error('Bias corrected data was not successfully applied to dataset.')
            return None
        else:
            logger.info(f'Bias for {date} successfully corrected.')

        # Close unnecessay datasets
        temp_masked.close()
        raw_masked.close()

    return template

def nan_test(var):
    """
    Returns location of Nans in debiased data.
    """
    
    fpath = glob.glob(str(current_dir / 'wrfout' / f'*{var}*.nc'))
    debiased = xr.open_dataset(fpath[0], decode_times = True)

    nan_data = debiased[var].where(debiased[var].isnull(), drop = True)

    return nan_data

def np_nan_loc(arr):
    """
    Find locations of NaNs in np.array.
    """
    return np.argwhere(np.isnan(arr))

# Catch silent errors and report to log file
@logger.catch 
def main(var, data_location, save = True):
    # Open datasets
    obs_path = glob.glob(str(data_location / 'gridMET' / f'*{var}*.nc'))
    obs = xr.open_dataset(obs_path[0], decode_times = True)

    raw_path = glob.glob(str(data_location / 'daily' / f'*{var}*.nc'))
    raw = xr.open_dataset(raw_path[0], decode_times = True)

    # Set location of interest as SLC airport
    lat = 40.788
    lon = -111.978

    # Pass datasets to debiaser
    test = ECDFM(obs, raw, var, lat, lon)

    if save:
        # Create output directory to store new cleaned files
        output_dir = current_dir / 'wrfout' 
        os.makedirs(output_dir, exist_ok = True) # Don't make if it already exists

        # Generate save name for debiased data
        out_path = os.path.join(output_dir, f'wrfout_GSLBIP_multimodel_ssp245_{var}_test.nc')

        # Save data to netCDF file
        test.to_netcdf(out_path)
        logger.success(f'File saved to: {out_path}')

    # Close out of data
    test.close()

# if __name__ == '__main__':

#     # Track program time in log files
#     start = time.perf_counter()
#     logger.info('Beginning execution.')

#     # Only inputs required
#     main(
#             var = 'tmmx',
#             data_location = current_dir,
#             save = True
#             )

#     # Report of runtime at completion 
#     logger.success(f'Debiasing process completed!')
#     logger.info(f'Total runtime: {time.perf_counter() - start:.4f}s')

#     # Force script to stop running once code is finished
#     sys.exit(0)


var = 'tmmx'
# Open datasets
obs_path = glob.glob(str(current_dir / 'gridMET' / f'*{var}*.nc'))
obs = xr.open_dataset(obs_path[0], decode_times = True)

raw_path = glob.glob(str(current_dir / 'daily' / f'*{var}*.nc'))
raw = xr.open_dataset(raw_path[0], decode_times = True)

# Set location of interest as SLC airport
lat = 40.788
lon = -111.978


# Slice raw data down to historical period
hist = raw.sel(time = slice('1985-01-01', '2014-12-31'))

# Select specific location of interest (choses nearest possible location)
obs_loc = loc_sel(obs, lat = lat, lon = lon)
logger.info(f'NaNs at selected obs location: {float(obs_loc[var].isnull().sum().values)}')
hist_loc = loc_sel(hist, lat = lat, lon = lon)
logger.info(f'NaNs at selected hist location: {float(hist_loc[var].isnull().sum().values)}')
raw_loc = loc_sel(raw, lat = lat, lon = lon)
logger.info(f'NaNs at selected raw location: {float(raw_loc[var].isnull().sum().values)}')
logger.info(raw_loc[var])

raw = raw_loc

# Make copy of raw_loc to plug debiased data into
template = raw_loc.copy()

# Create date range that spans a calendar year
date = pd.to_datetime('1985-12-31') 

# Apply bias correction for single date
logger.info(f'Date type: {type(date)}')
# debiased_doy = apply_to_date(obs_loc, hist_loc, raw_loc, var, date)

# Slice data into 31 day running windows centered on given date (note: 31 days is defualt setting)
obs_sliced = running_window_slice(obs, date)
logger.info(f'NaNs in obs running window: {float(obs_sliced[var].isnull().sum().values)}')

hist_sliced = running_window_slice(hist, date)
logger.info(f'NaNs in hist running window: {float(hist_sliced[var].isnull().sum().values)}')

raw_sliced = running_window_slice(raw, date)
logger.info(f'NaNs in raw running window: {float(raw_sliced[var].isnull().sum().values)}')
logger.info('Datasets sliced into running windows.')

# End function if NaNs are present in the source data
if obs_sliced[var].isnull().any() or hist_sliced[var].isnull().any() or raw_sliced[var].isnull().any():
    logger.error('NaNs detected in source data.')

# Create CDFs from sliced datasets
obs_cdf = ECDF(obs_sliced[var].values.flatten())
hist_cdf = ECDF(hist_sliced[var].values.flatten())
raw_cdf = ECDF(raw_sliced[var].values.flatten())

# Select day of interest and create mask
day = int(date.strftime('%d'))
month = int(date.strftime('%m'))
mask = (raw.time.dt.month == month) & (raw.time.dt.day == day)

# Skip dates that don't exist
if not mask.any():
    logger.warning(f'No date found for {date}.')

# Apply mask to dataset
raw_date = raw.where(mask, drop = True) # This should select day of interest across all years in ds
logger.info(f'Dataset sliced down to date of interest: {date}')
logger.info(raw_date)

18:28:44 |     INFO | plots.validation:loc_sel:line 89 - Selecting nearest location to 40.788, -111.978.
/uufs/chpc.utah.edu/common/home/strong-group7/sydney/miniforge3_envs/olympics/lib/python3.11/site-packages/xarray/core/dataarray.py:6318: FutureWarning: Behaviour of argmin/argmax with neither dim nor axis argument will change to return a dict of indices of each dimension. To get a single, flat index, please use np.argmin(da.data) or np.argmax(da.data) instead of da.argmin() or da.argmax().
  result = self.variable.argmin(dim, axis, keep_attrs, skipna)
18:28:44 |     INFO | plots.validation:loc_sel:line 100 - Point selected is <xarray.DataArray ()> Size: 8B
array(3916)
Coordinates:
    time     datetime64[ns] 8B 1985-01-01 from 40.788, -111.978.
18:28:44 |     INFO | plots.validation:loc_sel:line 102 - Min distance is located at 46, 43.
18:28:44 |     INFO | plots.validation:loc_sel:line 113 - New lat: 40.78413391113281
18:28:44 |     INFO | plots.validation:loc_sel:line 114 - New l

ValueError: zip() argument 2 is longer than argument 1

Connected to olympics (Python 3.11.15)

In [ ]:
"""
Author: Sydney Smith
Date Created: September 29, 2026
"""

from datetime import datetime
import gc
import glob
from loguru import logger
import numpy as np
import os
import pandas as pd
from pathlib import Path
from scipy.interpolate import interp1d
from statsmodels.distributions.empirical_distribution import ECDF, monotone_fn_inverter
import sys
import time
import xarray as xr

# ==================================
# - Establish Relative File Path - 
# ==================================

current_dir = Path(__file__).resolve().parent
sys.path.append(str(current_dir))

from plots.validation import running_window_slice, loc_sel

# ===================
# - Set Up Logger - 
# ==================

# Create directory for log files if it doesn't already exist
log_path = str(current_dir / 'log')
os.makedirs(log_path, exist_ok = True)

# String filename
log_filename = datetime.now().strftime('log/%Y-%m-%d_%H-%M-%S.log')

# Custom format for log prints
log_format = log_format = '<cyan>{time:HH:mm:ss}</cyan> | <level>{level:>8}</level> | <yellow>{name}</yellow>:<cyan>{function}</cyan>:line <magenta>{line}</magenta> - <level>{message}</level>'

# Removes the default stderr sink
logger.remove()  
# Anything above info to console
logger.add(sys.stderr, colorize = True, format = log_format, level = 'INFO') 
# Anything above debug to log file
logger.add(log_filename, colorize = False, format = log_format, level = 'DEBUG')
logger.debug(f'Log files saved to {log_path}')

# =====================
# ---- Functions ----
# =====================

def invert_ecdf(ecdf_obj):
    """
    Inverts a statsmodels ECDF object using its .y and .x attributes, regardless of if
    the values are monotonic or not, with safe extrapolation to prevent NaNs.
    """
    # ECDF from statsmodels sets first x value to -inf which causes NaNs for very small percentiles passed to the inverted function
    # Mask out any indefinite values to prevent possible NaNs
    finite_mask = np.isfinite(ecdf_obj.x)
    x_vals = ecdf_obj.x[finite_mask]
    y_vals = ecdf_obj.y[finite_mask]

    # Ensure x_vals are unique and strictly increasing for interp1d (monotonic)
    x_unique, indices = np.unique(x_vals, return_index=True)
    y_unique = y_vals[indices]

    return interp1d(
        y_unique, # Pass y values as x values
        x_unique, # And vice versa
        bounds_error = False, 
        fill_value = 'extrapolate'  # Prevents NaNs if raw percentiles push past bounds
    )

def apply_to_date(obs: xr.Dataset, hist, raw, var, date) -> np.ndarray:
    """
    Debiases data for a given date by constructing a CDF using a dataset restricted to be within a 
    running window. Date must be a valid date that exists within the dataset formatted as '%Y-%m-%d' 
    datetime object.
    """
    # TODO: check how running_window_slice handles leap years (try to set calendar year to 1988 instead since it's a leap year)

    # Slice data into 31 day running windows centered on given date (note: 31 days is defualt setting)
    obs_sliced = running_window_slice(obs, date)
    logger.info(f'NaNs in obs running window: {float(obs_sliced[var].isnull().sum().values)}')

    hist_sliced = running_window_slice(hist, date)
    logger.info(f'NaNs in hist running window: {float(hist_sliced[var].isnull().sum().values)}')

    raw_sliced = running_window_slice(raw, date)
    logger.info(f'NaNs in raw running window: {float(raw_sliced[var].isnull().sum().values)}')
    logger.info('Datasets sliced into running windows.')

    # End function if NaNs are present in the source data
    if obs_sliced[var].isnull().any() or hist_sliced[var].isnull().any() or raw_sliced[var].isnull().any():
        logger.error('NaNs detected in source data.')
        return NotImplementedError

    # Create CDFs from sliced datasets
    obs_cdf = ECDF(obs_sliced[var].values.flatten())
    hist_cdf = ECDF(hist_sliced[var].values.flatten())
    raw_cdf = ECDF(raw_sliced[var].values.flatten())

    # Select day of interest and create mask
    day = int(date.strftime('%d'))
    month = int(date.strftime('%m'))
    mask = (raw.time.dt.month == month) & (raw.time.dt.day == day)

    # Skip dates that don't exist
    if not mask.any():
        logger.warning(f'No date found for {date}.')
        return None
    
    # Apply mask to dataset
    raw_date = raw.where(mask, drop = True) # This should select day of interest across all years in ds
    logger.info(f'Dataset sliced down to date of interest: {date}')
    logger.info(raw_date)

    # Use raw_cdf like a function to find all of the percentiles in raw_date
    raw_percentiles = raw_cdf(raw_date[var].values)
    logger.info(f'Percentile values for raw data: {raw_percentiles}')
    logger.info(f'NaNs in raw_percentiles: {float(np.isnan(raw_percentiles).sum())}')

    # TODO: NaNs are appearing in obs and hist from this point on
    # Invert hist_cdf and obs_cdf to take a percentile and output a data point
    inv_obs = invert_ecdf(obs_cdf)
    inv_hist = invert_ecdf(hist_cdf)

    # Find data points for every percentile in raw_percentiles
    obs_vals = inv_obs(raw_percentiles)
    logger.info(f'NaNs in obs_vals: {float(np.isnan(obs_vals).sum())}')
    logger.info(f'Obs values for given percentiles: {obs_vals}')

    hist_vals = inv_hist(raw_percentiles)
    logger.info(f'NaNs in hist_vals: {float(np.isnan(hist_vals).sum())}')
    logger.info(f'Historical values for given percentiles: {hist_vals}')

    # Perform bias adjustment
    bias = hist_vals - obs_vals
    logger.info(f'NaNs in bias: {float(np.isnan(bias).sum())}')
    raw_bias_corrected = raw_date[var].values - bias

    # End script early if there are Nans in data
    if np.isnan(raw_bias_corrected).any():
        logger.error(f'NaNs in debiased data: {float(np.isnan(raw_bias_corrected).sum())}')
        logger.info(f'Nan error occured on {date}.')
        return None

    logger.info('Raw data successfully bias corrected!')
    logger.info(raw_bias_corrected)
    logger.info(f'Bias corrected max: {raw_bias_corrected.max()}')
    logger.info(f'Bias corrected min: {raw_bias_corrected.min()}')

    # Close out of unwanted datasets
    obs_sliced.close()
    hist_sliced.close()
    raw_sliced.close()
    raw_date.close()
    del obs_vals
    del hist_vals
    del raw_percentiles
    gc.collect()

    return raw_bias_corrected

def ECDFM(obs, raw, var, lat, lon):
    """
    Performs ECDFM quantile mapping debiasing process on given data based on data from a given histroical
    period and corresponding observational data. Raw data should include both historical and future data. 
    """

    # Slice raw data down to historical period
    hist = raw.sel(time = slice('1985-01-01', '2014-12-31'))
    
    # Select specific location of interest (choses nearest possible location)
    obs_loc = loc_sel(obs, lat = lat, lon = lon)
    logger.info(f'NaNs at selected obs location: {float(obs_loc[var].isnull().sum().values)}')
    hist_loc = loc_sel(hist, lat = lat, lon = lon)
    logger.info(f'NaNs at selected hist location: {float(hist_loc[var].isnull().sum().values)}')
    raw_loc = loc_sel(raw, lat = lat, lon = lon)
    logger.info(f'NaNs at selected raw location: {float(raw_loc[var].isnull().sum().values)}')
    logger.info(raw_loc[var])

    # Make copy of raw_loc to plug debiased data into
    template = raw_loc.copy()

    # Create date range that spans a calendar year
    dates = pd.date_range(start = '1985-01-01', end = '1985-12-31', freq = 'D') 

    for date in dates:
        # Apply bias correction for single date
        logger.info(f'Date type: {type(date)}')
        debiased_doy = apply_to_date(obs_loc, hist_loc, raw_loc, var, date)

        # Exit out of function if Nans occur
        if debiased_doy is None:
            logger.error(f'Error detected in dataset. See log above for more information.')
            return None

        # Pull month and day from date
        day = int(date.strftime('%d'))
        month = int(date.strftime('%m'))

        # Plug in debiased data
        mask = (template.time.dt.month == month) & (template.time.dt.day == day)
        template[var].data[mask] = debiased_doy 
        logger.info(f'Debiased data for {date} added to dataset!')

        # Back into data's bias to ensure that the bias corrected data doesn't equal to raw data
        temp_masked = template[var].where(mask, drop = True)
        raw_masked = raw[var].where(mask, drop = True)
        bias = raw_masked - temp_masked # Back into data's bias
        logger.info(f'bias type: {type(bias)}')

        # Ensure that bias is not all zeros
        if (bias == 0).all(): # Ensure it's not all zeros
            logger.error('Bias corrected data was not successfully applied to dataset.')
            return None
        else:
            logger.info(f'Bias for {date} successfully corrected.')

        # Close unnecessay datasets
        temp_masked.close()
        raw_masked.close()

    return template

def nan_test(var):
    """
    Returns location of Nans in debiased data.
    """
    
    fpath = glob.glob(str(current_dir / 'wrfout' / f'*{var}*.nc'))
    debiased = xr.open_dataset(fpath[0], decode_times = True)

    nan_data = debiased[var].where(debiased[var].isnull(), drop = True)

    return nan_data

def np_nan_loc(arr):
    """
    Find locations of NaNs in np.array.
    """
    return np.argwhere(np.isnan(arr))

# Catch silent errors and report to log file
@logger.catch 
def main(var, data_location, save = True):
    # Open datasets
    obs_path = glob.glob(str(data_location / 'gridMET' / f'*{var}*.nc'))
    obs = xr.open_dataset(obs_path[0], decode_times = True)

    raw_path = glob.glob(str(data_location / 'daily' / f'*{var}*.nc'))
    raw = xr.open_dataset(raw_path[0], decode_times = True)

    # Set location of interest as SLC airport
    lat = 40.788
    lon = -111.978

    # Pass datasets to debiaser
    test = ECDFM(obs, raw, var, lat, lon)

    if save:
        # Create output directory to store new cleaned files
        output_dir = current_dir / 'wrfout' 
        os.makedirs(output_dir, exist_ok = True) # Don't make if it already exists

        # Generate save name for debiased data
        out_path = os.path.join(output_dir, f'wrfout_GSLBIP_multimodel_ssp245_{var}_test.nc')

        # Save data to netCDF file
        test.to_netcdf(out_path)
        logger.success(f'File saved to: {out_path}')

    # Close out of data
    test.close()

# if __name__ == '__main__':

#     # Track program time in log files
#     start = time.perf_counter()
#     logger.info('Beginning execution.')

#     # Only inputs required
#     main(
#             var = 'tmmx',
#             data_location = current_dir,
#             save = True
#             )

#     # Report of runtime at completion 
#     logger.success(f'Debiasing process completed!')
#     logger.info(f'Total runtime: {time.perf_counter() - start:.4f}s')

#     # Force script to stop running once code is finished
#     sys.exit(0)


#TODO: why is it saying that 1985-12-31 does not exist in dataset -> from no mask in line 110

var = 'tmmx'
# Open datasets
obs_path = glob.glob(str(current_dir / 'gridMET' / f'*{var}*.nc'))
obs = xr.open_dataset(obs_path[0], decode_times = True)

raw_path = glob.glob(str(current_dir / 'daily' / f'*{var}*.nc'))
raw = xr.open_dataset(raw_path[0], decode_times = True)

# Set location of interest as SLC airport
lat = 40.788
lon = -111.978


# Slice raw data down to historical period
hist = raw.sel(time = slice('1985-01-01', '2014-12-31'))

# Select specific location of interest (choses nearest possible location)
obs_loc = loc_sel(obs, lat = lat, lon = lon)
logger.info(f'NaNs at selected obs location: {float(obs_loc[var].isnull().sum().values)}')
hist_loc = loc_sel(hist, lat = lat, lon = lon)
logger.info(f'NaNs at selected hist location: {float(hist_loc[var].isnull().sum().values)}')
raw_loc = loc_sel(raw, lat = lat, lon = lon)
logger.info(f'NaNs at selected raw location: {float(raw_loc[var].isnull().sum().values)}')
logger.info(raw_loc[var])

raw = raw_loc

# Make copy of raw_loc to plug debiased data into
template = raw_loc.copy()

# Create date range that spans a calendar year
date = pd.to_datetime('1985-12-31') 

# Apply bias correction for single date
logger.info(f'Date type: {type(date)}')
# debiased_doy = apply_to_date(obs_loc, hist_loc, raw_loc, var, date)

# Slice data into 31 day running windows centered on given date (note: 31 days is defualt setting)
obs_sliced = running_window_slice(obs, date)
logger.info(f'NaNs in obs running window: {float(obs_sliced[var].isnull().sum().values)}')

hist_sliced = running_window_slice(hist, date)
logger.info(f'NaNs in hist running window: {float(hist_sliced[var].isnull().sum().values)}')

raw_sliced = running_window_slice(raw, date)
logger.info(f'NaNs in raw running window: {float(raw_sliced[var].isnull().sum().values)}')
logger.info('Datasets sliced into running windows.')

# End function if NaNs are present in the source data
if obs_sliced[var].isnull().any() or hist_sliced[var].isnull().any() or raw_sliced[var].isnull().any():
    logger.error('NaNs detected in source data.')

# Create CDFs from sliced datasets
obs_cdf = ECDF(obs_sliced[var].values.flatten())
hist_cdf = ECDF(hist_sliced[var].values.flatten())
raw_cdf = ECDF(raw_sliced[var].values.flatten())

# Select day of interest and create mask
day = int(date.strftime('%d'))
month = int(date.strftime('%m'))
mask = (raw.time.dt.month == month) & (raw.time.dt.day == day)

# Skip dates that don't exist
if not mask.any():
    logger.warning(f'No date found for {date}.')

# Apply mask to dataset
raw_date = raw.where(mask, drop = True) # This should select day of interest across all years in ds
logger.info(f'Dataset sliced down to date of interest: {date}')
logger.info(raw_date)

10:50:28 |     INFO | plots.validation:loc_sel:line 89 - Selecting nearest location to 40.788, -111.978.
/uufs/chpc.utah.edu/common/home/strong-group7/sydney/miniforge3_envs/olympics/lib/python3.11/site-packages/xarray/core/dataarray.py:6318: FutureWarning: Behaviour of argmin/argmax with neither dim nor axis argument will change to return a dict of indices of each dimension. To get a single, flat index, please use np.argmin(da.data) or np.argmax(da.data) instead of da.argmin() or da.argmax().
  result = self.variable.argmin(dim, axis, keep_attrs, skipna)
10:50:28 |     INFO | plots.validation:loc_sel:line 100 - Point selected is <xarray.DataArray ()> Size: 8B
array(3916)
Coordinates:
    time     datetime64[ns] 8B 1985-01-01 from 40.788, -111.978.
10:50:28 |     INFO | plots.validation:loc_sel:line 102 - Min distance is located at 46, 43.
10:50:28 |     INFO | plots.validation:loc_sel:line 113 - New lat: 40.78413391113281
10:50:28 |     INFO | plots.validation:loc_sel:line 114 - New l

ValueError: zip() argument 2 is longer than argument 1

In [ ]:
"""
Author: Sydney Smith
Date Created: September 29, 2026
"""

from datetime import datetime
import gc
import glob
from loguru import logger
import numpy as np
import os
import pandas as pd
from pathlib import Path
from scipy.interpolate import interp1d
from statsmodels.distributions.empirical_distribution import ECDF, monotone_fn_inverter
import sys
import time
import xarray as xr

# ==================================
# - Establish Relative File Path - 
# ==================================

current_dir = Path(__file__).resolve().parent
sys.path.append(str(current_dir))

from plots.validation import running_window_slice, loc_sel

# ===================
# - Set Up Logger - 
# ==================

# Create directory for log files if it doesn't already exist
log_path = str(current_dir / 'log')
os.makedirs(log_path, exist_ok = True)

# String filename
log_filename = datetime.now().strftime('log/%Y-%m-%d_%H-%M-%S.log')

# Custom format for log prints
log_format = log_format = '<cyan>{time:HH:mm:ss}</cyan> | <level>{level:>8}</level> | <yellow>{name}</yellow>:<cyan>{function}</cyan>:line <magenta>{line}</magenta> - <level>{message}</level>'

# Removes the default stderr sink
logger.remove()  
# Anything above info to console
logger.add(sys.stderr, colorize = True, format = log_format, level = 'INFO') 
# Anything above debug to log file
logger.add(log_filename, colorize = False, format = log_format, level = 'DEBUG')
logger.debug(f'Log files saved to {log_path}')

# =====================
# ---- Functions ----
# =====================

def invert_ecdf(ecdf_obj):
    """
    Inverts a statsmodels ECDF object using its .y and .x attributes, regardless of if
    the values are monotonic or not, with safe extrapolation to prevent NaNs.
    """
    # ECDF from statsmodels sets first x value to -inf which causes NaNs for very small percentiles passed to the inverted function
    # Mask out any indefinite values to prevent possible NaNs
    finite_mask = np.isfinite(ecdf_obj.x)
    x_vals = ecdf_obj.x[finite_mask]
    y_vals = ecdf_obj.y[finite_mask]

    # Ensure x_vals are unique and strictly increasing for interp1d (monotonic)
    x_unique, indices = np.unique(x_vals, return_index=True)
    y_unique = y_vals[indices]

    return interp1d(
        y_unique, # Pass y values as x values
        x_unique, # And vice versa
        bounds_error = False, 
        fill_value = 'extrapolate'  # Prevents NaNs if raw percentiles push past bounds
    )

def apply_to_date(obs: xr.Dataset, hist, raw, var, date) -> np.ndarray:
    """
    Debiases data for a given date by constructing a CDF using a dataset restricted to be within a 
    running window. Date must be a valid date that exists within the dataset formatted as '%Y-%m-%d' 
    datetime object.
    """
    # TODO: check how running_window_slice handles leap years (try to set calendar year to 1988 instead since it's a leap year)

    # Slice data into 31 day running windows centered on given date (note: 31 days is defualt setting)
    obs_sliced = running_window_slice(obs, date)
    logger.info(f'NaNs in obs running window: {float(obs_sliced[var].isnull().sum().values)}')

    hist_sliced = running_window_slice(hist, date)
    logger.info(f'NaNs in hist running window: {float(hist_sliced[var].isnull().sum().values)}')

    raw_sliced = running_window_slice(raw, date)
    logger.info(f'NaNs in raw running window: {float(raw_sliced[var].isnull().sum().values)}')
    logger.info('Datasets sliced into running windows.')

    # End function if NaNs are present in the source data
    if obs_sliced[var].isnull().any() or hist_sliced[var].isnull().any() or raw_sliced[var].isnull().any():
        logger.error('NaNs detected in source data.')
        return NotImplementedError

    # Create CDFs from sliced datasets
    obs_cdf = ECDF(obs_sliced[var].values.flatten())
    hist_cdf = ECDF(hist_sliced[var].values.flatten())
    raw_cdf = ECDF(raw_sliced[var].values.flatten())

    # Select day of interest and create mask
    day = int(date.strftime('%d'))
    month = int(date.strftime('%m'))
    mask = (raw.time.dt.month == month) & (raw.time.dt.day == day)

    # Skip dates that don't exist
    if not mask.any():
        logger.warning(f'No date found for {date}.')
        return None
    
    # Apply mask to dataset
    raw_date = raw.where(mask, drop = True) # This should select day of interest across all years in ds
    logger.info(f'Dataset sliced down to date of interest: {date}')
    logger.info(raw_date)

    # Use raw_cdf like a function to find all of the percentiles in raw_date
    raw_percentiles = raw_cdf(raw_date[var].values)
    logger.info(f'Percentile values for raw data: {raw_percentiles}')
    logger.info(f'NaNs in raw_percentiles: {float(np.isnan(raw_percentiles).sum())}')

    # TODO: NaNs are appearing in obs and hist from this point on
    # Invert hist_cdf and obs_cdf to take a percentile and output a data point
    inv_obs = invert_ecdf(obs_cdf)
    inv_hist = invert_ecdf(hist_cdf)

    # Find data points for every percentile in raw_percentiles
    obs_vals = inv_obs(raw_percentiles)
    logger.info(f'NaNs in obs_vals: {float(np.isnan(obs_vals).sum())}')
    logger.info(f'Obs values for given percentiles: {obs_vals}')

    hist_vals = inv_hist(raw_percentiles)
    logger.info(f'NaNs in hist_vals: {float(np.isnan(hist_vals).sum())}')
    logger.info(f'Historical values for given percentiles: {hist_vals}')

    # Perform bias adjustment
    bias = hist_vals - obs_vals
    logger.info(f'NaNs in bias: {float(np.isnan(bias).sum())}')
    raw_bias_corrected = raw_date[var].values - bias

    # End script early if there are Nans in data
    if np.isnan(raw_bias_corrected).any():
        logger.error(f'NaNs in debiased data: {float(np.isnan(raw_bias_corrected).sum())}')
        logger.info(f'Nan error occured on {date}.')
        return None

    logger.info('Raw data successfully bias corrected!')
    logger.info(raw_bias_corrected)
    logger.info(f'Bias corrected max: {raw_bias_corrected.max()}')
    logger.info(f'Bias corrected min: {raw_bias_corrected.min()}')

    # Close out of unwanted datasets
    obs_sliced.close()
    hist_sliced.close()
    raw_sliced.close()
    raw_date.close()
    del obs_vals
    del hist_vals
    del raw_percentiles
    gc.collect()

    return raw_bias_corrected

def ECDFM(obs, raw, var, lat, lon):
    """
    Performs ECDFM quantile mapping debiasing process on given data based on data from a given histroical
    period and corresponding observational data. Raw data should include both historical and future data. 
    """

    # Slice raw data down to historical period
    hist = raw.sel(time = slice('1985-01-01', '2014-12-31'))
    
    # Select specific location of interest (choses nearest possible location)
    obs_loc = loc_sel(obs, lat = lat, lon = lon)
    logger.info(f'NaNs at selected obs location: {float(obs_loc[var].isnull().sum().values)}')
    hist_loc = loc_sel(hist, lat = lat, lon = lon)
    logger.info(f'NaNs at selected hist location: {float(hist_loc[var].isnull().sum().values)}')
    raw_loc = loc_sel(raw, lat = lat, lon = lon)
    logger.info(f'NaNs at selected raw location: {float(raw_loc[var].isnull().sum().values)}')
    logger.info(raw_loc[var])

    # Make copy of raw_loc to plug debiased data into
    template = raw_loc.copy()

    # Create date range that spans a calendar year
    dates = pd.date_range(start = '1985-01-01', end = '1985-12-31', freq = 'D') 

    for date in dates:
        # Apply bias correction for single date
        logger.info(f'Date type: {type(date)}')
        debiased_doy = apply_to_date(obs_loc, hist_loc, raw_loc, var, date)

        # Exit out of function if Nans occur
        if debiased_doy is None:
            logger.error(f'Error detected in dataset. See log above for more information.')
            return None

        # Pull month and day from date
        day = int(date.strftime('%d'))
        month = int(date.strftime('%m'))

        # Plug in debiased data
        mask = (template.time.dt.month == month) & (template.time.dt.day == day)
        template[var].data[mask] = debiased_doy 
        logger.info(f'Debiased data for {date} added to dataset!')

        # Back into data's bias to ensure that the bias corrected data doesn't equal to raw data
        temp_masked = template[var].where(mask, drop = True)
        raw_masked = raw[var].where(mask, drop = True)
        bias = raw_masked - temp_masked # Back into data's bias
        logger.info(f'bias type: {type(bias)}')

        # Ensure that bias is not all zeros
        if (bias == 0).all(): # Ensure it's not all zeros
            logger.error('Bias corrected data was not successfully applied to dataset.')
            return None
        else:
            logger.info(f'Bias for {date} successfully corrected.')

        # Close unnecessay datasets
        temp_masked.close()
        raw_masked.close()

    return template

def nan_test(var):
    """
    Returns location of Nans in debiased data.
    """
    
    fpath = glob.glob(str(current_dir / 'wrfout' / f'*{var}*.nc'))
    debiased = xr.open_dataset(fpath[0], decode_times = True)

    nan_data = debiased[var].where(debiased[var].isnull(), drop = True)

    return nan_data

def np_nan_loc(arr):
    """
    Find locations of NaNs in np.array.
    """
    return np.argwhere(np.isnan(arr))

# Catch silent errors and report to log file
@logger.catch 
def main(var, data_location, save = True):
    # Open datasets
    obs_path = glob.glob(str(data_location / 'gridMET' / f'*{var}*.nc'))
    obs = xr.open_dataset(obs_path[0], decode_times = True)

    raw_path = glob.glob(str(data_location / 'daily' / f'*{var}*.nc'))
    raw = xr.open_dataset(raw_path[0], decode_times = True)

    # Set location of interest as SLC airport
    lat = 40.788
    lon = -111.978

    # Pass datasets to debiaser
    test = ECDFM(obs, raw, var, lat, lon)

    if save:
        # Create output directory to store new cleaned files
        output_dir = current_dir / 'wrfout' 
        os.makedirs(output_dir, exist_ok = True) # Don't make if it already exists

        # Generate save name for debiased data
        out_path = os.path.join(output_dir, f'wrfout_GSLBIP_multimodel_ssp245_{var}_test.nc')

        # Save data to netCDF file
        test.to_netcdf(out_path)
        logger.success(f'File saved to: {out_path}')

    # Close out of data
    test.close()

# if __name__ == '__main__':

#     # Track program time in log files
#     start = time.perf_counter()
#     logger.info('Beginning execution.')

#     # Only inputs required
#     main(
#             var = 'tmmx',
#             data_location = current_dir,
#             save = True
#             )

#     # Report of runtime at completion 
#     logger.success(f'Debiasing process completed!')
#     logger.info(f'Total runtime: {time.perf_counter() - start:.4f}s')

#     # Force script to stop running once code is finished
#     sys.exit(0)


#TODO: why is it saying that 1985-12-31 does not exist in dataset -> from no mask in line 110

var = 'tmmx'
# Open datasets
obs_path = glob.glob(str(current_dir / 'gridMET' / f'*{var}*.nc'))
obs = xr.open_dataset(obs_path[0], decode_times = True)

raw_path = glob.glob(str(current_dir / 'daily' / f'*{var}*.nc'))
raw = xr.open_dataset(raw_path[0], decode_times = True)

# Set location of interest as SLC airport
lat = 40.788
lon = -111.978


# Slice raw data down to historical period
hist = raw.sel(time = slice('1985-01-01', '2014-12-31'))

# Select specific location of interest (choses nearest possible location)
obs_loc = loc_sel(obs, lat = lat, lon = lon)
logger.info(f'NaNs at selected obs location: {float(obs_loc[var].isnull().sum().values)}')
hist_loc = loc_sel(hist, lat = lat, lon = lon)
logger.info(f'NaNs at selected hist location: {float(hist_loc[var].isnull().sum().values)}')
raw_loc = loc_sel(raw, lat = lat, lon = lon)
logger.info(f'NaNs at selected raw location: {float(raw_loc[var].isnull().sum().values)}')
logger.info(raw_loc[var])

# Make copy of raw_loc to plug debiased data into
template = raw_loc.copy()

# Create date range that spans a calendar year
date = pd.to_datetime('1985-12-31') 

# Apply bias correction for single date
logger.info(f'Date type: {type(date)}')
# debiased_doy = apply_to_date(obs_loc, hist_loc, raw_loc, var, date)

# Slice data into 31 day running windows centered on given date (note: 31 days is defualt setting)
obs_sliced = running_window_slice(obs_loc, date)
logger.info(f'NaNs in obs running window: {float(obs_sliced[var].isnull().sum().values)}')

hist_sliced = running_window_slice(hist_loc, date)
logger.info(f'NaNs in hist running window: {float(hist_sliced[var].isnull().sum().values)}')

raw_sliced = running_window_slice(raw_loc, date)
logger.info(f'NaNs in raw running window: {float(raw_sliced[var].isnull().sum().values)}')
logger.info('Datasets sliced into running windows.')

# End function if NaNs are present in the source data
if obs_sliced[var].isnull().any() or hist_sliced[var].isnull().any() or raw_sliced[var].isnull().any():
    logger.error('NaNs detected in source data.')

# Create CDFs from sliced datasets
obs_cdf = ECDF(obs_sliced[var].values.flatten())
hist_cdf = ECDF(hist_sliced[var].values.flatten())
raw_cdf = ECDF(raw_sliced[var].values.flatten())

# Select day of interest and create mask
day = int(date.strftime('%d'))
month = int(date.strftime('%m'))
mask = (raw.time.dt.month == month) & (raw.time.dt.day == day)

# Skip dates that don't exist
if not mask.any():
    logger.warning(f'No date found for {date}.')

# Apply mask to dataset
raw_date = raw.where(mask, drop = True) # This should select day of interest across all years in ds
logger.info(f'Dataset sliced down to date of interest: {date}')
logger.info(raw_date)

10:51:22 |     INFO | plots.validation:loc_sel:line 89 - Selecting nearest location to 40.788, -111.978.
/uufs/chpc.utah.edu/common/home/strong-group7/sydney/miniforge3_envs/olympics/lib/python3.11/site-packages/xarray/core/dataarray.py:6318: FutureWarning: Behaviour of argmin/argmax with neither dim nor axis argument will change to return a dict of indices of each dimension. To get a single, flat index, please use np.argmin(da.data) or np.argmax(da.data) instead of da.argmin() or da.argmax().
  result = self.variable.argmin(dim, axis, keep_attrs, skipna)
10:51:22 |     INFO | plots.validation:loc_sel:line 100 - Point selected is <xarray.DataArray ()> Size: 8B
array(3916)
Coordinates:
    time     datetime64[ns] 8B 1985-01-01 from 40.788, -111.978.
10:51:22 |     INFO | plots.validation:loc_sel:line 102 - Min distance is located at 46, 43.
10:51:22 |     INFO | plots.validation:loc_sel:line 113 - New lat: 40.78413391113281
10:51:22 |     INFO | plots.validation:loc_sel:line 114 - New l

ValueError: size of dimension 'lat' on inputs was unexpectedly changed by applied function from 111 to 1. Only dimensions specified in ``exclude_dims`` with xarray.apply_ufunc are allowed to change size. The data returned was:

array([], shape=(0, 1, 1), dtype=float32)

In [ ]:
raw

<xarray.Dataset> Size: 5GB
Dimensions:  (time: 41888, lat: 111, lon: 90, south_north: 111, west_east: 90)
Coordinates:
  * time     (time) datetime64[ns] 335kB 1985-01-01 1985-01-02 ... 2099-12-30
    lat      (time, south_north, west_east) float32 2GB ...
    lon      (time, south_north, west_east) float32 2GB ...
Dimensions without coordinates: south_north, west_east
Data variables:
    tmmx     (time, lat, lon) float32 2GB ...
Attributes: (12/155)
    TITLE:                            OUTPUT FROM WRF V4.6.0 MODEL
    START_DATE:                      1984-01-01_06:00:00
    SIMULATION_START_DATE:           1984-01-01_06:00:00
    WEST-EAST_GRID_DIMENSION:        91
    SOUTH-NORTH_GRID_DIMENSION:      112
    BOTTOM-TOP_GRID_DIMENSION:       39
    ...                              ...
    ISLAKE:                          21
    ISICE:                           15
    ISURBAN:                         13
    ISOILWATER:                      14
    HYBRID_OPT:                      2
    ETAC:                            0.2

In [ ]:
raw_loc

<xarray.Dataset> Size: 2GB
Dimensions:  (time: 41888, lat: 111, lon: 90)
Coordinates:
  * time     (time) datetime64[ns] 335kB 1985-01-01 1985-01-02 ... 2099-12-30
    lat      (time) float32 168kB ...
    lon      (time) float32 168kB ...
Data variables:
    tmmx     (time, lat, lon) float32 2GB 268.8 269.5 269.9 ... 272.6 272.2
Attributes: (12/155)
    TITLE:                            OUTPUT FROM WRF V4.6.0 MODEL
    START_DATE:                      1984-01-01_06:00:00
    SIMULATION_START_DATE:           1984-01-01_06:00:00
    WEST-EAST_GRID_DIMENSION:        91
    SOUTH-NORTH_GRID_DIMENSION:      112
    BOTTOM-TOP_GRID_DIMENSION:       39
    ...                              ...
    ISLAKE:                          21
    ISICE:                           15
    ISURBAN:                         13
    ISOILWATER:                      14
    HYBRID_OPT:                      2
    ETAC:                            0.2

In [ ]:
raw_sliced

<xarray.Dataset> Size: 138MB
Dimensions:  (time: 3450, lat: 111, lon: 90)
Coordinates:
  * time     (time) datetime64[ns] 28kB 1985-01-01 1985-01-02 ... 2099-12-30
    lat      (time) float32 14kB 40.78 40.78 40.78 40.78 ... 40.78 40.78 40.78
    lon      (time) float32 14kB -112.0 -112.0 -112.0 ... -112.0 -112.0 -112.0
Data variables:
    tmmx     (time, lat, lon) float32 138MB 268.8 269.5 269.9 ... 272.6 272.2
Attributes: (12/155)
    TITLE:                            OUTPUT FROM WRF V4.6.0 MODEL
    START_DATE:                      1984-01-01_06:00:00
    SIMULATION_START_DATE:           1984-01-01_06:00:00
    WEST-EAST_GRID_DIMENSION:        91
    SOUTH-NORTH_GRID_DIMENSION:      112
    BOTTOM-TOP_GRID_DIMENSION:       39
    ...                              ...
    ISLAKE:                          21
    ISICE:                           15
    ISURBAN:                         13
    ISOILWATER:                      14
    HYBRID_OPT:                      2
    ETAC:                            0.2

In [ ]:
raw

<xarray.Dataset> Size: 5GB
Dimensions:  (time: 41888, lat: 111, lon: 90, south_north: 111, west_east: 90)
Coordinates:
  * time     (time) datetime64[ns] 335kB 1985-01-01 1985-01-02 ... 2099-12-30
    lat      (time, south_north, west_east) float32 2GB ...
    lon      (time, south_north, west_east) float32 2GB ...
Dimensions without coordinates: south_north, west_east
Data variables:
    tmmx     (time, lat, lon) float32 2GB ...
Attributes: (12/155)
    TITLE:                            OUTPUT FROM WRF V4.6.0 MODEL
    START_DATE:                      1984-01-01_06:00:00
    SIMULATION_START_DATE:           1984-01-01_06:00:00
    WEST-EAST_GRID_DIMENSION:        91
    SOUTH-NORTH_GRID_DIMENSION:      112
    BOTTOM-TOP_GRID_DIMENSION:       39
    ...                              ...
    ISLAKE:                          21
    ISICE:                           15
    ISURBAN:                         13
    ISOILWATER:                      14
    HYBRID_OPT:                      2
    ETAC:                            0.2